<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/01_financial_news_scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

วิธีการเชื่อมต่อ runtime local

รัน text นี้ใน terminal

jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --NotebookApp.port_retention=False --NotebookApp.disable_check_xsrf=True

In [ ]:
!pip install -q curl-cffi beautifulsoup4 lxml pandas tqdm playwright requests
!playwright install -q chromium

## 📂 กลุ่มที่ 1: WordPress REST API Retrieval (Kaohoon, Money&Banking, ThaiPR, HoonSmart)

กลุ่มนี้ใช้วิธีการดึงข้อมูลที่มีประสิทธิภาพสูงสุด โดยการเชื่อมต่อกับ **REST API Endpoint** ของ WordPress (`/wp-json/wp/v2/posts`) ซึ่งช่วยให้ได้ข้อมูลในรูปแบบโครงสร้าง JSON ที่สมบูรณ์โดยไม่ต้อง Parse HTML ซับซ้อน

### **คุณสมบัติทางเทคนิค:**
- **Efficiency:** ดึงข้อมูลได้สูงสุด 100 รายการต่อ 1 Request
- **Date Filtering:** ใช้ Parameter `after` และ `before` เพื่อจำกัดช่วงเวลาปี 2023-2024 ตั้งแต่ต้นทาง
- **Data Quality:** ข้อมูล Title และ Content ถูกดึงมาจาก Field `rendered` ซึ่งมีความแม่นยำสูง

### **1.1 Kaohoon (ข่าวหุ้น)**
- **Module:** `curl_cffi`, `BeautifulSoup`
- **Logic:** เน้นการดึงข้อมูลผ่าน API โดยใช้ `curl_cffi` เพื่อเลียนแบบ Browser Fingerprint ป้องกันการถูกจำกัดสิทธิ์ (Rate Limit)
- **Cleansing:** ล้าง HTML entities ที่มากับ API (เช่น `&#8211;`) และรวม Paragraphs เข้าด้วยกัน
- **Variable:** `after_date`, `before_date` กำหนดขอบเขตเวลา ISO 8601

In [ ]:
#Kaohoon
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_wp_api_sniper(start_year=2023, end_year=2024):
    print(f"\n🚀 [WP API SNIPER] เริ่มต้นสูบข้อมูลจาก API ปี {start_year}-{end_year}")
    print(f"{'='*75}")

    # กำหนดเป้าหมายระดับ "หวานหมู"
    targets = {
        "Kaohoon": "https://www.kaohoon.com"
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    # 📅 กำหนดขอบเขตเวลา (ISO 8601 Format สำหรับ WP API)
    after_date = f"{start_year}-01-01T00:00:00"
    before_date = f"{end_year}-12-31T23:59:59"

    for site_name, base_url in targets.items():
        print(f"\n🏢 กำลังโจมตีเป้าหมาย: {site_name}")

        filename = f"dataset_{site_name}_API_{start_year}_{end_year}.jsonl"
        with open(filename, "w", encoding="utf-8") as f: pass

        api_url = f"{base_url}/wp-json/wp/v2/posts"
        page = 1
        total_saved = 0

        while True:
            print(f"  📖 กำลังดึงข้อมูลหน้าที่ {page} (หน้าละ 100 ข่าว)...", end="\r")

            # 🎯 อาวุธลับ: ใส่ Parameter ฟิลเตอร์วันที่แบบแนบเนียน
            params = {
                "per_page": 100,  # ขอข้อมูลสูงสุด 100 ชิ้นต่อการดึง 1 ครั้ง
                "page": page,
                "after": after_date,
                "before": before_date
            }

            try:
                resp = cureq.get(api_url, params=params, headers=headers, impersonate="chrome", timeout=20)

                # ถ้า Status เป็น 400 ใน WP API มักแปลว่า "หมดหน้าแล้ว (Page out of bounds)"
                if resp.status_code == 400:
                    print(f"\n      🏁 ดึงข้อมูลครบทุกหน้าแล้วสำหรับ {site_name}!")
                    break
                elif resp.status_code != 200:
                    print(f"\n      🔴 หยุดชะงัก (Status: {resp.status_code})")
                    break

                data = resp.json()

                if not data or len(data) == 0:
                    print(f"\n      🏁 ไม่พบข้อมูลเพิ่มแล้ว สิ้นสุดการค้นหา!")
                    break

                # ลุยแกะกล่อง JSON
                for item in data:
                    # ข้อมูลที่ได้จะเป็น HTML ปนมา เราต้องใช้ BeautifulSoup ล้างอีกรอบ
                    raw_title = item.get('title', {}).get('rendered', '')
                    raw_content = item.get('content', {}).get('rendered', '')
                    link = item.get('link', '')
                    pub_date = item.get('date', '') # Format: YYYY-MM-DDTHH:MM:SS

                    # 🧹 ซักล้าง Title (แปลงพวกโค้ดแปลกๆ เช่น &#8211; ให้เป็นตัวอักษรปกติ)
                    clean_title = BeautifulSoup(raw_title, 'html.parser').get_text(strip=True)
                    clean_title = re.sub(r'["\'“”‘’]', '', clean_title)

                    # 🧹 ซักล้าง Content
                    content_soup = BeautifulSoup(raw_content, 'html.parser')
                    paragraphs = content_soup.find_all('p')

                    # บางเว็บอาจไม่ได้ใช้ <p> เสมอไป ถ้าหา <p> ไม่เจอ ให้ดึง Text ทั้งหมดแทน
                    if paragraphs:
                        clean_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
                    else:
                        clean_text = content_soup.get_text(strip=True)

                    clean_text = re.sub(r'https?://\S+|www\.\S+', '', clean_text)
                    clean_text = re.sub(r'["\'“”‘’]', '', clean_text)
                    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

                    if len(clean_text) < 50: continue

                    record = {
                        "date": pub_date.split('T')[0], # ตัดเอาเฉพาะ YYYY-MM-DD
                        "url": link,
                        "title": clean_title,
                        "content": clean_text
                    }

                    with open(filename, "a", encoding="utf-8") as f:
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")

                    total_saved += 1

                page += 1
                time.sleep(1) # พักเซิร์ฟเวอร์เขาสัก 1 วินาที ป้องกันโดนแบน IP

            except Exception as e:
                print(f"\n      ❌ Error หน้าที่ {page}: {e}")
                break

        print(f"\n🎉 ปิดจ๊อบ {site_name}! กวาดข้อมูลสำเร็จ {total_saved} ข่าว")

if __name__ == "__main__":
    scrape_wp_api_sniper(start_year=2023, end_year=2024)


🚀 [WP API SNIPER] เริ่มต้นสูบข้อมูลจาก API ปี 2023-2024

🏢 กำลังโจมตีเป้าหมาย: Kaohoon

      🏁 ดึงข้อมูลครบทุกหน้าแล้วสำหรับ Kaohoon!

🎉 ปิดจ๊อบ Kaohoon! กวาดข้อมูลสำเร็จ 39759 ข่าว

🏢 กำลังโจมตีเป้าหมาย: Money_Banking

      🏁 ดึงข้อมูลครบทุกหน้าแล้วสำหรับ Money_Banking!

🎉 ปิดจ๊อบ Money_Banking! กวาดข้อมูลสำเร็จ 31433 ข่าว


### **1.2 Money & Banking (การเงินธนาคาร)**
- **Module:** `categories` parameter ใน API
- **Logic:** ใช้การล็อกรหัสหมวดหมู่ (Category ID: 102, 105) ซึ่งเป็นหมวดการเงินและธนาคารโดยเฉพาะ ทำให้ไม่ต้องเสียเวลาดึงข่าวหมวดอื่น
- **Cleansing:** กำจัดเครื่องหมายคำพูดพิเศษ และกรองข่าวที่เนื้อหาสั้นเกินไป (< 50 chars)

In [ ]:
#Money&Banking
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_money_banking_category_locked(start_year=2023, end_year=2024):
    print(f"\n🚀 [WP API SNIPER V.3] ล็อกเป้าหมายด้วย Category ID (ปี {start_year}-{end_year})")
    print(f"{'='*80}")

    site_name = "Money_Banking"
    base_url = "https://moneyandbanking.co.th"

    # 🎯 อัปเกรด: ใส่รหัส ID ที่ได้จากขั้นตอนที่ 1 ตรงนี้ (คั่นด้วยลูกน้ำ)
    target_category_ids = "102,105"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    after_date = f"{start_year}-01-01T00:00:00"
    before_date = f"{end_year}-12-31T23:59:59"

    filename = f"dataset_{site_name}_API_CategoryLocked_{start_year}_{end_year}.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    api_url = f"{base_url}/wp-json/wp/v2/posts"
    page = 1
    total_saved = 0

    while True:
        print(f"  📖 กำลังดึงข้อมูลหน้าที่ {page} (หน้าละ 100 ข่าว)...", end="\r")

        # 🎯 อัปเกรด: สั่งเซิร์ฟเวอร์กรองหมวดหมู่ให้ตั้งแต่ต้นทาง!
        params = {
            "per_page": 100,
            "page": page,
            "after": after_date,
            "before": before_date,
            "categories": target_category_ids # << หัวใจสำคัญอยู่ตรงนี้ครับ!
        }

        try:
            resp = cureq.get(api_url, params=params, headers=headers, impersonate="chrome", timeout=20)

            if resp.status_code == 400:
                print(f"\n      🏁 ดึงข้อมูลครบทุกหน้าแล้ว!")
                break
            elif resp.status_code != 200:
                print(f"\n      🔴 หยุดชะงัก (Status: {resp.status_code})")
                break

            data = resp.json()
            if not data or len(data) == 0:
                print(f"\n      🏁 ไม่พบข้อมูลเพิ่มแล้ว สิ้นสุดการค้นหา!")
                break

            for item in data:
                # ⚡ ไม่ต้องใช้ตะแกรงร่อนคีย์เวิร์ดแล้ว เพราะ API กรองมาให้แม่น 100%
                raw_title = item.get('title', {}).get('rendered', '')
                link = item.get('link', '')

                clean_title = BeautifulSoup(raw_title, 'html.parser').get_text(strip=True)
                clean_title = re.sub(r'["\'“”‘’]', '', clean_title)

                raw_content = item.get('content', {}).get('rendered', '')
                pub_date = item.get('date', '')

                content_soup = BeautifulSoup(raw_content, 'html.parser')
                paragraphs = content_soup.find_all('p')

                if paragraphs:
                    clean_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
                else:
                    clean_text = content_soup.get_text(strip=True)

                clean_text = re.sub(r'https?://\S+|www\.\S+', '', clean_text)
                clean_text = re.sub(r'["\'“”‘’]', '', clean_text)
                clean_text = re.sub(r'\s+', ' ', clean_text).strip()

                if len(clean_text) < 50: continue

                record = {
                    "date": pub_date.split('T')[0],
                    "url": link,
                    "title": clean_title,
                    "content": clean_text
                }

                with open(filename, "a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

                total_saved += 1

            page += 1
            time.sleep(1) # พักเซิร์ฟเวอร์เขาสัก 1 วินาที

        except Exception as e:
            print(f"\n      ❌ Error หน้าที่ {page}: {e}")
            break

    print(f"\n🎉 ปิดจ๊อบ Money & Banking! กวาดข้อมูลเป้าหมายสำเร็จ {total_saved} ข่าว")

if __name__ == "__main__":
    scrape_money_banking_category_locked(start_year=2023, end_year=2024)


🚀 [WP API SNIPER V.3] ล็อกเป้าหมายด้วย Category ID (ปี 2023-2024)

      🏁 ดึงข้อมูลครบทุกหน้าแล้ว!

🎉 ปิดจ๊อบ Money & Banking! กวาดข้อมูลเป้าหมายสำเร็จ 7899 ข่าว


### **1.3 ThaiPR (ไทยพีอาร์)**
- **Module:** Exact Link Filtering
- **Logic:** แม้ดึงผ่าน API แต่มีการเพิ่ม Filter ชั้นที่สองเพื่อตรวจสอบว่า URL ต้องมี `/finance/` เท่านั้น เพื่อป้องกันข่าวประชาสัมพันธ์ทั่วไปที่ไม่เกี่ยวข้องกับเศรษฐกิจปนเข้ามา
- **Cleansing:** ตัดส่วนที่เป็น Footer ของข่าว PR เช่น "แท็ก:", "ข้อมูลเพิ่มเติม"

In [ ]:
#Thaipr
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_thaipr_finance_exact_tag(start_year=2023, end_year=2024):
    print(f"\n🚀 [WP API SNIPER] ล็อกเป้า ThaiPR (เฉพาะหมวด /finance/ เป๊ะๆ) ปี {start_year}-{end_year}")
    print(f"{'='*80}")

    site_name = "ThaiPR"
    base_url = "https://www.thaipr.net"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    after_date = f"{start_year}-01-01T00:00:00"
    before_date = f"{end_year}-12-31T23:59:59"

    filename = f"dataset_{site_name}_API_FinanceExact_{start_year}_{end_year}.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    api_url = f"{base_url}/wp-json/wp/v2/posts"
    page = 1
    total_saved = 0

    while True:
        print(f"  📖 กำลังดึงข้อมูลหน้าที่ {page} (หน้าละ 100 ข่าว)...", end="\r")

        params = {
            "per_page": 100,
            "page": page,
            "after": after_date,
            "before": before_date
        }

        try:
            resp = cureq.get(api_url, params=params, headers=headers, impersonate="chrome", timeout=20)

            if resp.status_code == 400:
                print(f"\n      🏁 ดึงข้อมูลครบทุกหน้าแล้ว!")
                break
            elif resp.status_code != 200:
                print(f"\n      🔴 หยุดชะงัก (Status: {resp.status_code})")
                break

            data = resp.json()
            if not data or len(data) == 0:
                print(f"\n      🏁 ไม่พบข้อมูลเพิ่มแล้ว สิ้นสุดการค้นหา!")
                break

            for item in data:
                link = item.get('link', '')

                # ==========================================
                # 🎯 อัปเกรด: ล็อกเป้า /finance/ แบบปิดหัวปิดท้าย
                # ==========================================
                if '/finance/' not in link.lower():
                    continue # เตะทิ้งทันที! (พวก /finance_en/ หรือ /refinance/ จะโดนคัดออกหมด)
                # ==========================================

                raw_title = item.get('title', {}).get('rendered', '')
                clean_title = BeautifulSoup(raw_title, 'html.parser').get_text(strip=True)
                clean_title = re.sub(r'["\'“”‘’]', '', clean_title)

                raw_content = item.get('content', {}).get('rendered', '')
                pub_date = item.get('date', '')

                content_soup = BeautifulSoup(raw_content, 'html.parser')
                paragraphs = content_soup.find_all('p')

                if paragraphs:
                    clean_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
                else:
                    clean_text = content_soup.get_text(strip=True)

                # 🧹 ระบบ Cleansing ทำความสะอาดเนื้อหา
                clean_text = re.sub(r'https?://\S+|www\.\S+', '', clean_text)
                clean_text = re.sub(r'["\'“”‘’]', '', clean_text)
                clean_text = re.sub(r'\s+', ' ', clean_text).strip()

                # ตัดคำขยะท้ายข่าว PR
                tail_keywords = ["แท็ก:", "ข้อมูลเพิ่มเติม", "ที่มา:"]
                for kw in tail_keywords:
                    if kw in clean_text:
                        clean_text = clean_text.split(kw)[0].strip()

                if len(clean_text) < 50: continue

                record = {
                    "date": pub_date.split('T')[0],
                    "url": link,
                    "title": clean_title,
                    "content": clean_text
                }

                with open(filename, "a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

                total_saved += 1

            page += 1
            time.sleep(1) # พักเซิร์ฟเวอร์เขาสัก 1 วินาที

        except Exception as e:
            print(f"\n      ❌ Error หน้าที่ {page}: {e}")
            break

    print(f"\n🎉 ปิดจ๊อบ ThaiPR! กวาดข่าว /finance/ (ภาษาไทยล้วน) มาได้ {total_saved} ข่าว")
    print(f"📁 ไฟล์พร้อมใช้งานที่: {filename}")

if __name__ == "__main__":
    scrape_thaipr_finance_exact_tag(start_year=2023, end_year=2024)


🚀 [WP API SNIPER] ล็อกเป้า ThaiPR (เฉพาะหมวด /finance/ เป๊ะๆ) ปี 2023-2024

      🏁 ดึงข้อมูลครบทุกหน้าแล้ว!

🎉 ปิดจ๊อบ ThaiPR! กวาดข่าว /finance/ (ภาษาไทยล้วน) มาได้ 10305 ข่าว
📁 ไฟล์พร้อมใช้งานที่: dataset_ThaiPR_API_FinanceExact_2023_2024.jsonl


### **1.4 HoonSmart (หุ้นสมาร์ท)**
- **Module:** `_fields` parameter optimization
- **Logic:** ระบุ Field ที่ต้องการ (`date,link,title,content`) ใน Request เพื่อลดปริมาณ Data Transfer
- **Cleansing:** มี Regex พิเศษเพื่อลบลายน้ำประจำเว็บ `HoonSmart.com >>` ออกจากต้นประโยคทุกข่าว

In [ ]:
#HoonSmart
import requests
import json
from bs4 import BeautifulSoup
import time
import re

def scrape_hoonsmart_production_2023_2024():
    # 📅 ตั้งค่าช่วงเวลา: 1 มกราคม 2023 - 31 ธันวาคม 2024
    start_date = "2023-01-01T00:00:00"
    end_date = "2024-12-31T23:59:59"
    api_url = "https://hoonsmart.com/wp-json/wp/v2/posts"
    page = 1

    # ชื่อไฟล์สำหรับรวบรวม Dataset
    filename = "dataset_hoonsmart_2023_2024_ultraclean.jsonl"
    total_saved = 0

    print(f"\n🚀 [PRODUCTION MODE] เดินเครื่องดูดข้อมูล HoonSmart ปี 2023-2024")
    print(f"{'='*70}")

    while True:
        params = {
            "after": start_date,
            "before": end_date,
            "per_page": 100, # ดึงสูงสุด 100 ข่าวต่อหน้า
            "page": page,
            "_fields": "date,link,title,content"
        }

        try:
            response = requests.get(api_url, params=params, timeout=20)

            # เช็คว่าทะลุหน้าสุดท้ายหรือยัง
            if response.status_code == 400:
                break
            if response.status_code != 200:
                print(f"\n❌ พบปัญหาการเชื่อมต่อ Status: {response.status_code}")
                break

            data = response.json()
            if not data:
                break

            with open(filename, "a", encoding="utf-8") as f:
                for item in data:
                    # 1. ถอดแท็ก HTML
                    soup = BeautifulSoup(item['content']['rendered'], 'html.parser')
                    text = soup.get_text(separator=' ', strip=True)

                    # 2. 🧹 ลบลายน้ำต้นข่าว (HoonSmart.com>>)
                    text = re.sub(r'^HoonSmart\.com\s*>>\s*', '', text, flags=re.IGNORECASE)

                    # 3. 🧹 ลบขีดยาวทุกตระกูล (-, _, —, –, =) ที่เรียงติดกัน 5 ตัวขึ้นไป
                    text = re.sub(r'[-_—–=]{5,}', '', text)

                    # 4. 🧹 จัดระเบียบช่องว่าง
                    clean_content = re.sub(r'\s+', ' ', text).strip()

                    # 🛑 5. กรองข่าวขยะ/โพสต์คั่นหน้า (สั้นกว่า 50 ตัวอักษร)
                    if len(clean_content) < 50:
                        continue

                    # 6. ล้างแท็ก HTML จากหัวข้อข่าว
                    clean_title = BeautifulSoup(item['title']['rendered'], 'html.parser').get_text(strip=True)

                    # จัดทรงข้อมูลเตรียมเทรนโมเดล
                    record = {
                        "date": item['date'],
                        "url": item['link'],
                        "title": clean_title,
                        "content": clean_content
                    }
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    total_saved += 1

            print(f"💾 โหลดหน้า {page} สำเร็จ | สะสมแล้ว {total_saved} ข่าว...", end="\r")

            page += 1
            time.sleep(0.5) # พักเซิร์ฟเวอร์นิดหน่อย

        except Exception as e:
            print(f"\n❌ Error ที่หน้า {page}: {e}")
            break

    print(f"\n\n🎉 ภารกิจสมบูรณ์! กวาดข้อมูลช่วงปี 2023-2024 มาได้ทั้งหมด {total_saved} ข่าว")
    print(f"📁 ไฟล์พร้อมใช้งาน: {filename}")

if __name__ == "__main__":
    # เคลียร์ไฟล์เก่า (ถ้ามี) ก่อนเริ่มรันของใหม่ เพื่อไม่ให้ข้อมูลซ้ำ
    open("dataset_hoonsmart_2023_2024_ultraclean.jsonl", "w", encoding="utf-8").close()
    scrape_hoonsmart_production_2023_2024()


🚀 [PRODUCTION MODE] เดินเครื่องดูดข้อมูล HoonSmart ปี 2023-2024


🎉 ภารกิจสมบูรณ์! กวาดข้อมูลช่วงปี 2023-2024 มาได้ทั้งหมด 7889 ข่าว
📁 ไฟล์พร้อมใช้งาน: dataset_hoonsmart_2023_2024_ultraclean.jsonl


## 📂 กลุ่มที่ 2: XML Sitemap & Static HTML Parsing (Infoquest, TNN, Amarin, Thairath, PPTV)

กลุ่มนี้ใช้วิธีการ **Sitemap Sniping** โดยการอ่านไฟล์ XML ดัชนีของเว็บไซต์ เพื่อหา URL ข่าวตามวันที่ที่ต้องการ แล้วจึงเข้าไปดึงเนื้อหา (Scraping) ทีละลิงก์

### **คุณสมบัติทางเทคนิค:**
- **Targeting:** เข้าถึงข่าวเก่าย้อนหลังได้แม่นยำผ่าน Monthly/Daily Sitemap
- **Reliability:** ลดการดึงข้อมูลที่ไร้ประสิทธิภาพด้วยการกรอง URL Keyword ก่อนเริ่ม Scraping

### **2.1 Infoquest (อินโฟเควสท์)**
- **Logic:** ใช้การสแกน XML Sitemap ลำดับที่ 60-140 ซึ่งเป็นช่วงดัชนีข่าวปี 2023-2024
- **Cleansing:** ตัดขยะจากระบบ Dataxet เช่น "อัปเดตข่าวสารด้านเศรษฐกิจ..." และตัดวงเล็บวันที่ท้ายข่าว เช่น `(26 ม.ค. 66)` ออกเพื่อความสะอาดของ Text

In [ ]:
#Infoquest
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_infoquest_locked_production(start_year=2023, end_year=2024):
    print(f"\n🚀 [INFOQUEST PRODUCTION] โหมดล็อกเป้าแผนที่ & ล้างขยะ Dataxet")
    print(f"{'='*70}")

    filename = f"dataset_infoquest_{start_year}_{end_year}_Clean_Locked.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    # 🗺️ 1. ล็อกเป้าแฟ้ม Sitemap
    # ข่าวปี 2023-2024 คาดว่าจะอยู่ในช่วงแฟ้มที่ 60 ถึง 140 (อิงจากแฟ้ม 73 ที่เราเจอ)
    # เราสร้างรายชื่อ URL รอไว้เลย ไม่ต้องไปโหลด Index ให้เสียเวลา
    locked_sitemaps = [f"https://www.infoquest.co.th/post-sitemap{i}.xml" for i in range(60, 140)]

    target_links = set()

    print(f"🔍 1. สแกนลิงก์เป้าหมายจากแฟ้มที่ล็อกไว้ (แฟ้มที่ 60-139)...")
    for sm_url in locked_sitemaps:
        try:
            sm_resp = cureq.get(sm_url, impersonate="chrome", timeout=15)
            if sm_resp.status_code != 200:
                continue

            sm_soup = BeautifulSoup(sm_resp.text, 'xml')
            urls = sm_soup.find_all('loc')

            found_in_this_file = 0
            for url_tag in urls:
                link = url_tag.text
                year_match = re.search(r'/(\d{4})/', link)
                if year_match and start_year <= int(year_match.group(1)) <= end_year:
                    target_links.add(link)
                    found_in_this_file += 1

            if found_in_this_file > 0:
                print(f"  - สแกน {sm_url.split('/')[-1]}: พบเป้าหมาย {found_in_this_file} ลิงก์", end="\r")

        except Exception:
            pass

    print(f"\n\n🎯 ล็อกเป้าหมายข่าวปี {start_year}-{end_year} ได้ทั้งหมด {len(target_links)} ลิงก์")
    if not target_links: return

    print("\n🚜 2. สตาร์ทเครื่องสูบและล้างขยะ Dataxet...")
    total_saved = 0

    for link in target_links:
        try:
            art_resp = cureq.get(link, impersonate="chrome", timeout=15)
            if art_resp.status_code == 200:
                art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                paragraphs = art_soup.find_all('p')
                raw_content = ' '.join([p.get_text(strip=True) for p in paragraphs])

                # 🧹 1. กรอง URL และลายน้ำ
                clean_content = re.sub(r'https?://\S+|www\.\S+', '', raw_content)
                clean_content = re.sub(r'(?:สำนักข่าวอินโฟเควสท์|โดย สำนักข่าวอินโฟเควสท์|อินโฟเควสท์|InfoQuest)', '', clean_content, flags=re.IGNORECASE)

                # 🧹 2. [อัปเกรด] ตัดขยะโฆษณา Dataxet ท้ายข่าว
                # ใช้ .split() หั่นข้อความทิ้งตั้งแต่คำแรกที่เจอ
                if "อัปเดตข่าวสารด้านเศรษฐกิจ" in clean_content:
                    clean_content = clean_content.split("อัปเดตข่าวสารด้านเศรษฐกิจ")[0]
                if "บริษัท ดาต้าเซ็ต จำกัด" in clean_content:
                    clean_content = clean_content.split("บริษัท ดาต้าเซ็ต จำกัด")[0]

                # 🧹 3. [อัปเกรด] ตัดวงเล็บวันที่ท้ายข่าว เช่น (26 ม.ค. 66)
                # Regex นี้จะมองหาวงเล็บที่มี ตัวเลข+เว้นวรรค+เดือนย่อ+ปี ที่อยู่ "ท้ายสุด" ของประโยค
                clean_content = re.sub(r'\(\d+\s*[ก-ฮ]+\.[ก-ฮ]+\.\s*\d+\)\s*$', '', clean_content).strip()

                # เคลียร์ช่องว่างส่วนเกิน
                clean_content = re.sub(r'\s+', ' ', clean_content).strip()

                if len(clean_content) < 50: continue

                title_tag = art_soup.find('h1') or art_soup.find('title')
                clean_title = title_tag.get_text(strip=True) if title_tag else ""

                date_tag = art_soup.find('meta', property='article:published_time')
                year_match = re.search(r'/(\d{4})/', link)
                fallback_year = year_match.group(1) if year_match else str(end_year)
                pub_date = date_tag['content'][:10] if date_tag else f"{fallback_year}-01-01"

                record = {
                    "date": pub_date,
                    "url": link,
                    "title": clean_title,
                    "content": clean_content
                }

                with open(filename, "a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

                total_saved += 1
                print(f"💾 ดูดและทำความสะอาดแล้ว {total_saved}/{len(target_links)} ข่าว...", end="\r")

        except Exception:
            continue

    print(f"\n\n🎉 ภารกิจสำเร็จ! กวาดข้อมูลพรีเมียม (ไร้ขยะ) มาได้ {total_saved} ข่าว")
    print(f"📁 ไฟล์พร้อมใช้งานที่: {filename}")

if __name__ == "__main__":
    scrape_infoquest_locked_production()


🚀 [INFOQUEST PRODUCTION] โหมดล็อกเป้าแผนที่ & ล้างขยะ Dataxet
🔍 1. สแกนลิงก์เป้าหมายจากแฟ้มที่ล็อกไว้ (แฟ้มที่ 60-139)...


🎯 ล็อกเป้าหมายข่าวปี 2023-2024 ได้ทั้งหมด 30194 ลิงก์

🚜 2. สตาร์ทเครื่องสูบและล้างขยะ Dataxet...
💾 ดูดและทำความสะอาดแล้ว 20557/30194 ข่าว...

🎉 ภารกิจสำเร็จ! กวาดข้อมูลพรีเมียม (ไร้ขยะ) มาได้ 20557 ข่าว
📁 ไฟล์พร้อมใช้งานที่: dataset_infoquest_2023_2024_Clean_Locked.jsonl


### **2.2 TNN Wealth (ทีเอ็นเอ็น)**
- **Logic:** ดึงข้อมูลผ่าน Monthly Sitemap (`sitemap_news_YYYYMM.xml`) เจาะจงเฉพาะ Path `/wealth/`
- **Cleansing:** มีระบบ **Ultra Refine** ตัดส่วน Ticker ข่าวที่ปนมากับเนื้อหา และตัด Footer "TNN ทันโลก ทันเศรษฐกิจ"

In [ ]:
#TNN
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re

def ultra_refine_text(text):
    if not text: return ""
    # 1. ตัดส่วนที่เป็น 'หัวข่าวแนะนำ' หรือ 'Ticker' ต้นประโยค
    ticker_patterns = [
        r'^.*?"ทรัมป์".*?"ซัมซุง".*?"Starbucks".*?สู้เงินเฟ้อ',
        r'^.*?Clean Energy:.*?YLG มองทองคำยังขาขึ้น',
        r'^.*?ราคาน้ำนมโลกลดลงต่อเนื่อง.*?จุดเสี่ยงเขย่าตลาดทองคำ'
    ]
    for pattern in ticker_patterns:
        text = re.sub(pattern, '', text).strip()

    # 2. ตัดส่วนท้ายที่เป็น Footer บริษัท
    footer_markers = [
        "TNN ทันโลก ทันเศรษฐกิจ", "อาคาร 101 ทรู ดิจิทัล",
        "Email :[email protected]", "WorldWealthHealthTechEarth"
    ]
    for marker in footer_markers:
        if marker in text: text = text.split(marker)[0].strip()

    # 3. ลบเครื่องหมายคำพูด (") และจัดระเบียบช่องว่าง
    text = text.replace('"', '')
    text = ' '.join(text.split())
    return text

def scrape_and_clean_tnn_production(start_year=2023, end_year=2024):
    print(f"🚀 [TNN UNIFIED PRODUCTION] Target Years: {start_year}-{end_year}")
    session = cureq.Session(impersonate="chrome", timeout=30)
    filename = f"dataset_tnn_wealth_{start_year}_{end_year}_ultra_refined.jsonl"

    # สร้างรายชื่อเดือนเป้าหมาย
    target_months = [f"{y}{m:02d}" for y in range(end_year, start_year - 1, -1) for m in range(12, 0, -1)]
    total_saved = 0

    with open(filename, "w", encoding="utf-8") as f:
        for yyyymm in target_months:
            sitemap_url = f"https://www.tnnthailand.com/sitemap_news_{yyyymm}.xml"
            try:
                resp = session.get(sitemap_url)
                if resp.status_code != 200: continue

                sm_soup = BeautifulSoup(resp.text, 'xml')
                links = [loc.text for loc in sm_soup.find_all('loc') if '/wealth/' in loc.text]
                print(f"\n📂 {yyyymm}: พบ {len(links)} ลิงก์")

                for link in links:
                    try:
                        art_resp = session.get(link)
                        if art_resp.status_code == 200:
                            soup = BeautifulSoup(art_resp.text, 'html.parser')

                            # สกัดเนื้อหาเบื้องต้นจาก p
                            content_div = soup.find('div', class_=re.compile(r'news-detail-content|content-detail|detail-box', re.I)) or soup.find('article') or soup
                            paragraphs = [p.get_text(strip=True) for p in content_div.find_all('p') if len(p.get_text(strip=True)) > 20]
                            raw_text = " ".join(paragraphs)

                            # Apply Ultra Refined Cleaning
                            clean_title = ultra_refine_text(soup.find('h1').get_text(strip=True) if soup.find('h1') else "")
                            clean_body = ultra_refine_text(raw_text)

                            if len(clean_body) > 150:
                                meta_date = soup.find('meta', property='article:published_time')
                                record = {
                                    "date": meta_date['content'][:10] if meta_date else "",
                                    "url": link,
                                    "title": clean_title,
                                    "content": clean_body
                                }
                                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                total_saved += 1
                                print(f"  ✅ บันทึกแล้ว: {total_saved}", end="\r")
                    except: continue
                time.sleep(1)
            except Exception as e:
                print(f"Error {yyyymm}: {e}")

    print(f"\n\n🎉 เสร็จสมบูรณ์! รวมทั้งหมด: {total_saved} รายการ")
    print(f"📁 ไฟล์ผลลัพธ์: {filename}")

if __name__ == '__main__':
    scrape_and_clean_tnn_production()

🚀 [TNN UNIFIED PRODUCTION] Target Years: 2023-2024

📂 202412: พบ 539 ลิงก์

📂 202411: พบ 525 ลิงก์

📂 202410: พบ 643 ลิงก์

📂 202409: พบ 579 ลิงก์

📂 202408: พบ 501 ลิงก์

📂 202407: พบ 197 ลิงก์

📂 202406: พบ 147 ลิงก์

📂 202405: พบ 122 ลิงก์

📂 202404: พบ 123 ลิงก์

📂 202403: พบ 136 ลิงก์

📂 202402: พบ 128 ลิงก์

📂 202401: พบ 112 ลิงก์

📂 202312: พบ 109 ลิงก์

📂 202311: พบ 104 ลิงก์

📂 202310: พบ 126 ลิงก์

📂 202309: พบ 127 ลิงก์

📂 202308: พบ 123 ลิงก์

📂 202307: พบ 209 ลิงก์

📂 202306: พบ 194 ลิงก์

📂 202305: พบ 206 ลิงก์

📂 202304: พบ 190 ลิงก์

📂 202303: พบ 290 ลิงก์

📂 202302: พบ 238 ลิงก์

📂 202301: พบ 297 ลิงก์


🎉 เสร็จสมบูรณ์! รวมทั้งหมด: 5936 รายการ
📁 ไฟล์ผลลัพธ์: dataset_tnn_wealth_2023_2024_ultra_refined.jsonl


### **2.3 Amarin Spotlight (อมรินทร์)**
- **Logic:** ใช้ **Sniper Method** กรองลิงก์ที่มีคำว่า `finance` ตั้งแต่ระดับ Sitemap
- **Cleansing:** ลบส่วนประกอบ UI เช่น "Live Spotlight ข่าว", "ทุบโต๊ะข่าว" และตัดท้ายบทความด้วยคำสำคัญ เช่น "ที่มา :"

In [ ]:
#Amarin
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_amarin_finance_fast(start_year=2023, end_year=2024):
    print(f"\n🚀 [AMARIN FINANCE SNIPER V.2 + ADVANCED CLEANSING] สกัดข่าวการเงิน (ปี {start_year}-{end_year})")
    print(f"{'='*80}")

    filename = f"dataset_amarintv_finance_FastURL_{start_year}_{end_year}.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    total_saved = 0

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    # 🎯 คีย์เวิร์ดกวาด URL (ถ้ามีคำเหล่านี้ในลิงก์ ดึงทันที!)
    url_keywords = [
        'finance'
    ]

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            sitemap_url = f"https://www.amarintv.com/sitemap/spotlight-{year}-{month:02d}.xml"
            print(f"\n📂 กางแผนที่: {year}-{month:02d} ...", end=" ")

            try:
                sm_resp = cureq.get(sitemap_url, headers=headers, impersonate="chrome", timeout=15)

                if sm_resp.status_code != 200:
                    print("🟡 ไม่มีแฟ้มนี้ (ข้าม)")
                    continue

                soup = BeautifulSoup(sm_resp.text, 'xml')
                urls = soup.find_all('loc')

                if not urls:
                    print("⚪ แฟ้มว่างเปล่า")
                    continue

                # ==========================================
                # ⚡ ตะแกรงร่อนความเร็วแสง: คัดเฉพาะ URL เป้าหมาย
                # ==========================================
                finance_links = []
                for url_tag in urls:
                    link = url_tag.text.lower()
                    if any(kw in link for kw in url_keywords):
                        finance_links.append(url_tag.text)
                # ==========================================

                if not finance_links:
                    print("⚪ ไม่มีข่าวการเงินในเดือนนี้")
                    continue

                print(f"🎯 ล็อกเป้า {len(finance_links)} ข่าว! โจมตี...")

                for link in finance_links:
                    try:
                        art_resp = cureq.get(link, headers=headers, impersonate="chrome", timeout=15)
                        if art_resp.status_code == 200:
                            art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                            # ==========================================
                            # 🚜 ดึงเนื้อหาและระบบ Cleansing ขั้นสูง
                            # ==========================================
                            paragraphs = art_soup.find_all('p')

                            # 1. คัดกรองแต่ละย่อหน้าก่อนนำมารวมกัน
                            valid_paragraphs = []
                            for p in paragraphs:
                                text = p.get_text(strip=True)

                                # ข้ามย่อหน้าสั้นๆ หรือเมนูนำทาง
                                if len(text) < 20:
                                    continue
                                if text.startswith("Live Spotlight ข่าว") or text.startswith("อ่านข่าวเพิ่มเติม"):
                                    continue

                                valid_paragraphs.append(text)

                            raw_content = ' '.join(valid_paragraphs)

                            # 2. หั่นขยะส่วนหัว (ถ้าหลุดรอดมาได้)
                            header_garbage = "Live Spotlight ข่าว รายการวาไรตี้ รายการทุบโต๊ะข่าว ยานยนต์ รายการตามอำเภอจาน ข่าวต่างประเทศ"
                            if raw_content.startswith(header_garbage):
                                raw_content = raw_content[len(header_garbage):].strip()

                            # 3. หั่นขยะส่วนท้าย (ใช้ดาบซามูไรตัดทิ้งตั้งแต่คีย์เวิร์ดพวกนี้ไปจนสุด)
                            tail_keywords = [
                                "อ่านข่าวเพิ่มเติม",
                                "บทความที่เกี่ยวข้อง",
                                "ที่มา :",
                                "ที่มา:",
                                "คลิกอ่านข่าว",
                                "ติดตามข่าวสาร",
                                "ต้นข่าว"
                            ]

                            for kw in tail_keywords:
                                if kw in raw_content:
                                    raw_content = raw_content.split(kw)[0]

                            # 4. ทำความสะอาดขั้นสุดท้าย
                            clean_content = re.sub(r'https?://\S+|www\.\S+', '', raw_content)
                            clean_content = re.sub(r'["\'“”‘’]', '', clean_content)
                            clean_content = re.sub(r'\s+', ' ', clean_content).strip()

                            if len(clean_content) < 50: continue
                            # ==========================================

                            title_tag = art_soup.find('h1') or art_soup.find('title')
                            clean_title = re.sub(r'["\'“”‘’]', '', title_tag.get_text(strip=True)) if title_tag else ""

                            record = {
                                "date": f"{year}-{month:02d}",
                                "url": link,
                                "title": clean_title,
                                "content": clean_content
                            }

                            with open(filename, "a", encoding="utf-8") as f:
                                f.write(json.dumps(record, ensure_ascii=False) + "\n")

                            total_saved += 1
                            print(f"    💸 ดูดสำเร็จ {total_saved} ข่าว...", end="\r")
                            time.sleep(0.1)

                    except Exception: pass

            except Exception as e:
                print(f"❌ Error การโหลดแฟ้ม: {e}")

    print(f"\n\n🎉 ภารกิจสำเร็จ! สกัดเป้าหมายพร้อมคลีนเนื้อหาได้ทั้งหมด {total_saved} ข่าว")
    print(f"📁 ไฟล์พร้อมใช้งานที่: {filename}")

if __name__ == "__main__":
    scrape_amarin_finance_fast(start_year=2023, end_year=2024)


🚀 [AMARIN FINANCE SNIPER V.2 + ADVANCED CLEANSING] สกัดข่าวการเงิน (ปี 2023-2024)

📂 กางแผนที่: 2023-01 ... 🎯 ล็อกเป้า 57 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 56 ข่าว...
📂 กางแผนที่: 2023-02 ... 🎯 ล็อกเป้า 45 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 101 ข่าว...
📂 กางแผนที่: 2023-03 ... 🎯 ล็อกเป้า 63 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 164 ข่าว...
📂 กางแผนที่: 2023-04 ... 🎯 ล็อกเป้า 20 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 184 ข่าว...
📂 กางแผนที่: 2023-05 ... 🎯 ล็อกเป้า 46 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 230 ข่าว...
📂 กางแผนที่: 2023-06 ... 🎯 ล็อกเป้า 61 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 289 ข่าว...
📂 กางแผนที่: 2023-07 ... 🎯 ล็อกเป้า 64 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 353 ข่าว...
📂 กางแผนที่: 2023-08 ... 🎯 ล็อกเป้า 63 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 416 ข่าว...
📂 กางแผนที่: 2023-09 ... 🎯 ล็อกเป้า 61 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 477 ข่าว...
📂 กางแผนที่: 2023-10 ... 🎯 ล็อกเป้า 72 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 547 ข่าว...
📂 กางแผนที่: 2023-11 ... 🎯 ล็อกเป้า 95 ข่าว! โจมตี...
    💸 ดูดสำเร็จ 641 ข่าว...
📂 กางแผนที่: 20

### **2.4 Thairath Investment (ไทยรัฐ)**
- **Logic:** กรอง URL ภายใต้หมวด `/money/investment/` และต้องไม่มีคำว่า `gold` เพื่อแยกข่าวราคาทองรายวันออกจากข่าวการเงินเชิงลึก
- **Cleansing:** ลบเครื่องหมายอัญประกาศซ้ำซ้อน และ Normalize ช่องว่าง

In [ ]:
#Thairath
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re

def scrape_thairath_investment_strict(start_year=2023, end_year=2024):
    print(f"\n🚀 [STRICT INVESTMENT] กำลังเริ่มสกัดข่าว (ปี {start_year}-{end_year})")

    file_path = f"thairath_strict_investment_{start_year}_{end_year}.jsonl"
    # เคลียร์ไฟล์เก่าทิ้งก่อนเริ่มใหม่เพื่อให้มั่นใจว่าไม่มีข่าว gold ปน
    with open(file_path, "w", encoding="utf-8") as f: pass

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    total_saved = 0

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            sitemap_url = f"https://www.thairath.co.th/sitemap-monthly-{year}-{month:02d}.xml"
            try:
                resp = cureq.get(sitemap_url, headers=headers, impersonate="chrome", timeout=30)
                if resp.status_code != 200: continue

                soup = BeautifulSoup(resp.text, 'xml')
                # กรองเฉพาะ /money/investment/ และต้องไม่มีคำว่า gold ใน URL
                urls = [
                    loc.text for loc in soup.find_all('loc')
                    if '/money/investment/' in loc.text and 'gold' not in loc.text.lower()
                ]

                if urls:
                    print(f"\n📂 ตรวจเช็ค: {year}-{month:02d} | พบข่าวลงทุน (ไม่เอา Gold) {len(urls)} ลิงก์")

                    for url in urls:
                        try:
                            art_resp = cureq.get(url, headers=headers, impersonate="chrome", timeout=15)
                            if art_resp.status_code != 200: continue

                            art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                            # ดึง Title และลบเครื่องหมาย "
                            title_raw = art_soup.find('h1').get_text(strip=True) if art_soup.find('h1') else ""
                            title = title_raw.replace('"', '').replace('"', '')

                            # ดึง Content และลบเครื่องหมาย "
                            paragraphs = art_soup.find_all(['p', 'article'])
                            content_raw = ' '.join([p.get_text(strip=True) for p in paragraphs if len(p.get_text()) > 20])
                            content = re.sub(r'\s+', ' ', content_raw).strip()
                            content = content.replace('"', '').replace('"', '')

                            if len(content) > 100:
                                record = {"date": f"{year}-{month:02d}", "url": url, "title": title, "content": content}
                                with open(file_path, "a", encoding="utf-8") as f:
                                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                total_saved += 1
                                print(f"    ✅ บันทึกแล้ว {total_saved} ข่าว...", end="\r")
                        except: continue
            except: continue

    print(f"\n\n🎉 Finished! Total Investment News (No Gold): {total_saved}")
    print(f"File saved at: {file_path}")

if __name__ == '__main__':
    scrape_thairath_investment_strict(2023, 2024)


🚀 [STRICT INVESTMENT] กำลังเริ่มสกัดข่าว (ปี 2023-2024)

📂 ตรวจเช็ค: 2023-01 | พบข่าวลงทุน (ไม่เอา Gold) 116 ลิงก์

📂 ตรวจเช็ค: 2023-02 | พบข่าวลงทุน (ไม่เอา Gold) 129 ลิงก์

📂 ตรวจเช็ค: 2023-03 | พบข่าวลงทุน (ไม่เอา Gold) 153 ลิงก์

📂 ตรวจเช็ค: 2023-04 | พบข่าวลงทุน (ไม่เอา Gold) 122 ลิงก์

📂 ตรวจเช็ค: 2023-05 | พบข่าวลงทุน (ไม่เอา Gold) 149 ลิงก์

📂 ตรวจเช็ค: 2023-06 | พบข่าวลงทุน (ไม่เอา Gold) 178 ลิงก์

📂 ตรวจเช็ค: 2023-07 | พบข่าวลงทุน (ไม่เอา Gold) 176 ลิงก์

📂 ตรวจเช็ค: 2023-08 | พบข่าวลงทุน (ไม่เอา Gold) 189 ลิงก์

📂 ตรวจเช็ค: 2023-09 | พบข่าวลงทุน (ไม่เอา Gold) 185 ลิงก์

📂 ตรวจเช็ค: 2023-10 | พบข่าวลงทุน (ไม่เอา Gold) 174 ลิงก์

📂 ตรวจเช็ค: 2023-11 | พบข่าวลงทุน (ไม่เอา Gold) 168 ลิงก์

📂 ตรวจเช็ค: 2023-12 | พบข่าวลงทุน (ไม่เอา Gold) 139 ลิงก์

📂 ตรวจเช็ค: 2024-01 | พบข่าวลงทุน (ไม่เอา Gold) 161 ลิงก์

📂 ตรวจเช็ค: 2024-02 | พบข่าวลงทุน (ไม่เอา Gold) 157 ลิงก์

📂 ตรวจเช็ค: 2024-03 | พบข่าวลงทุน (ไม่เอา Gold) 137 ลิงก์

📂 ตรวจเช็ค: 2024-04 | พบข่าวลงทุน (ไม่เอา Gold) 122 ลิงก์

### **2.5 PPTV Wealth (พีพีทีวี)**
- **Logic:** ใช้ระบบ **Time Machine V.3** วนลูปรายวัน (Daily Loop) เพื่อเข้าถึง Sitemap ทุกวันในปี 2023-2024
- **Cleansing:** ตัด Header แบนเนอร์คุกกี้ และส่วนของอัตราแลกเปลี่ยนถัวเฉลี่ยท้ายข่าว

In [ ]:
#PPTV
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
from datetime import date, timedelta
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

# ปิดแจ้งเตือนกวนใจ
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def generate_date_range(start_date, end_date):
    """ฟังก์ชันสร้างปฏิทินรายวันตามช่วงเวลาที่กำหนด"""
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + timedelta(n)

def scrape_pptv_production_super_clean(start_year=2023, end_year=2024):
    print(f"\n🚀 [PPTV TIME MACHINE V.3] ปฏิบัติการเจาะคลังข่าว หุ้น & การเงิน (ปี {start_year}-{end_year})")
    print(f"{'='*70}")

    filename = f"dataset_pptv_{start_year}_{end_year}_SuperClean_Production.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    # 📅 สร้างปฏิทินตั้งแต่วันแรกจนถึงวันสุดท้ายของช่วงปีที่กำหนด
    s_date = date(start_year, 1, 1)
    e_date = date(end_year, 12, 31)
    date_list = list(generate_date_range(s_date, e_date))

    total_saved = 0
    # 🎯 โฟกัสเฉพาะหมวดหุ้นและการเงิน
    target_keywords = ['/wealth/stock-investment/', '/wealth/monetary/']

    print(f"📅 เริ่มแกะรอยแผนที่ Sitemap ทั้งหมด {len(date_list)} วัน...")

    for i, current_date in enumerate(date_list):
        date_str = current_date.strftime("%Y-%m-%d")
        sitemap_url = f"https://www.pptvhd36.com/sitemap-{date_str}.xml"

        print(f"\n🔍 [วันที่ {i+1}/{len(date_list)}] กำลังสแกนแผนที่วันที่: {date_str}")

        try:
            sm_resp = cureq.get(sitemap_url, impersonate="chrome", timeout=15)
            if sm_resp.status_code != 200:
                print(f"  🟡 ข้าม: ไม่มีไฟล์ Sitemap ของวันนี้")
                continue

            sm_soup = BeautifulSoup(sm_resp.text, 'xml')
            urls = sm_soup.find_all('loc')

            # คัดกรองเอาเฉพาะข่าวที่เกี่ยวกับหุ้น/การเงิน
            day_target_links = [u.text for u in urls if any(kw in u.text for kw in target_keywords)]

            if not day_target_links:
                 print(f"  ⚪ ข้าม: วันนี้ไม่มีข่าวในหมวดเป้าหมาย")
                 continue

            print(f"  🎯 พบข่าวเป้าหมาย {len(day_target_links)} ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...")

            for link in day_target_links:
                try:
                    art_resp = cureq.get(link, impersonate="chrome", timeout=15)
                    if art_resp.status_code == 200:
                        art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                        paragraphs = art_soup.find_all('p')
                        raw_content = ' '.join([p.get_text(strip=True) for p in paragraphs])

                        # ==========================================
                        # 🧹 เครื่องซักผ้าดิจิทัล (Data Cleansing)
                        # ==========================================
                        # 0. ลบ URL ที่แทรกในเนื้อหา
                        clean_content = re.sub(r'https?://\S+|www\.\S+', '', raw_content)

                        # 1. หั่นหัว: ตัดแบนเนอร์คุกกี้และผู้เขียน
                        if "โดยPPTV Online" in clean_content:
                            clean_content = clean_content.split("โดยPPTV Online")[-1]
                        elif "คุณสามารถปรับแต่งคุกกี้ได้ที่นี่" in clean_content:
                            clean_content = clean_content.split("คุณสามารถปรับแต่งคุกกี้ได้ที่นี่")[-1]

                        # 2. หั่นหาง: ตัดข่าวแนะนำและอัตราแลกเปลี่ยนท้ายข่าว
                        tail_garbage = [
                            "คอนเทนต์แนะนำ",
                            "อัตราแลกเปลี่ยนถัวเฉลี่ยถ่วงน้ำหนัก",
                            "ข้อมูลจาก :",
                            "เรียบเรียงจาก"
                        ]
                        for g in tail_garbage:
                            if g in clean_content:
                                clean_content = clean_content.split(g)[0]

                        # 3. ล้างไส้ใน: ลบคำขยะ
                        inline_garbage = [
                            "เนื้อหาคัดสรรคุณภาพ",
                            "คริปโต เอ็กซ์เชนจ์",
                            "PPTV Online"
                        ]
                        for g in inline_garbage:
                            clean_content = clean_content.replace(g, "")

                        title_tag = art_soup.find('h1') or art_soup.find('title')
                        clean_title = title_tag.get_text(strip=True) if title_tag else ""

                        # 4. ล้างอักขระพิเศษและเครื่องหมายคำพูด (ตัด Noise)
                        clean_title = re.sub(r'["\'“”‘’]', '', clean_title)
                        clean_content = re.sub(r'["\'“”‘’]', '', clean_content)

                        # 5. เคลียร์ช่องว่างส่วนเกินรอบสุดท้าย
                        clean_content = re.sub(r'\s+', ' ', clean_content).strip()
                        clean_title = re.sub(r'\s+', ' ', clean_title).strip()
                        # ==========================================

                        # ถ้าเนื้อหาสั้นเกินไป (ข่าวโดนลบไปแล้ว หรือมีแต่รูปภาพ) ให้ข้าม
                        if len(clean_content) < 50:
                            continue

                        record = {
                            "date": date_str,
                            "url": link,
                            "title": clean_title,
                            "content": clean_content
                        }

                        with open(filename, "a", encoding="utf-8") as f:
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")

                        total_saved += 1
                        print(f"    ✨ คลีนเนียนกริ๊บและเซฟแล้ว {total_saved} ข่าว...", end="\r")
                        time.sleep(0.2) # หน่วงเวลาเล็กน้อยป้องกันโดนบล็อก IP

                except Exception:
                    pass # ลิงก์เสียให้ข้ามไปเลย

        except Exception as e:
            print(f"  ❌ Error การโหลด Sitemap: {e}")

    print(f"\n\n🎉 ภารกิจสำเร็จลุล่วง! กวาดข้อมูลข่าวหุ้นและการเงินมาได้ทั้งหมด {total_saved} ข่าว")
    print(f"📁 ไฟล์พร้อมใช้งานที่: {filename}")

if __name__ == "__main__":
    # เริ่มเดินเครื่องไทม์แมชชีน!
    scrape_pptv_production_super_clean(start_year=2023, end_year=2024)


🚀 [PPTV TIME MACHINE V.3] ปฏิบัติการเจาะคลังข่าว หุ้น & การเงิน (ปี 2023-2024)
📅 เริ่มแกะรอยแผนที่ Sitemap ทั้งหมด 731 วัน...

🔍 [วันที่ 1/731] กำลังสแกนแผนที่วันที่: 2023-01-01
  🎯 พบข่าวเป้าหมาย 3 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...

🔍 [วันที่ 2/731] กำลังสแกนแผนที่วันที่: 2023-01-02
  🎯 พบข่าวเป้าหมาย 1 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...
    ✨ คลีนเนียนกริ๊บและเซฟแล้ว 4 ข่าว...
🔍 [วันที่ 3/731] กำลังสแกนแผนที่วันที่: 2023-01-03
  🎯 พบข่าวเป้าหมาย 4 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...
    ✨ คลีนเนียนกริ๊บและเซฟแล้ว 8 ข่าว...
🔍 [วันที่ 4/731] กำลังสแกนแผนที่วันที่: 2023-01-04
  🎯 พบข่าวเป้าหมาย 6 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...

🔍 [วันที่ 5/731] กำลังสแกนแผนที่วันที่: 2023-01-05
  🎯 พบข่าวเป้าหมาย 7 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...

🔍 [วันที่ 6/731] กำลังสแกนแผนที่วันที่: 2023-01-06
  🎯 พบข่าวเป้าหมาย 7 ชิ้น! เริ่มกระบวนการสูบและทำความสะอาด...

🔍 [วันที่ 7/731] กำลังสแกนแผนที่วันที่: 2023-01-07
  🎯 พบข่าวเป้าหมาย 1 ชิ้น! เริ่มกระบวนการสูบและทำความสะอา

## 📂 กลุ่มที่ 3: Custom AJAX & Deep Scraping (Thunhoon, Share2trade, Hoonvision, Prachachat)

กลุ่มนี้จัดการกับเว็บไซต์ที่มีระบบโครงสร้างซับซ้อน หรือต้องอาศัยการส่งค่าผ่าน AJAX เพื่อดึงข้อมูลออกมา

### **คุณสมบัติทางเทคนิค:**
- **Dynamic Handling:** จัดการการเรียกข้อมูลแบบ Asynchronous
- **Deep Crawling:** วนลูปหน้า Archive (Pagination) เพื่อย้อนกลับไปถึงปี 2023

### **3.1 Share2trade (แชร์ทูเทรด)**
- **Logic:** แบ่งการดึงเป็น 3 หมวดหลัก (Talk of the Town, Bulletin Board, Stock Daily Journal) โดยกำหนดหน้าเริ่มต้น (Start Page) ต่างกันเพื่อให้ครอบคลุมช่วงปี 2023
- **Tech:** ใช้ `Session` ในการคงสภาพการเชื่อมต่อ และมีระบบตรวจจับปี พ.ศ. (2566/2567) เพื่อแปลงเป็น ค.ศ.
- **Cleansing:** กรอง Paragraph ที่มีความยาวสั้นกว่า 10 ตัวอักษรออก (ซึ่งมักเป็นชื่อภาพหรือโฆษณาแทรก)

In [ ]:
#Share2trade
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings

warnings.filterwarnings("ignore")

def scrape_share2trade_production_v11(start_year=2023, end_year=2024):
    print(f"\n🚀 [SHARE2TRADE V.11] แยกหน้าเริ่มต้นตามหมวดหมู่")
    print(f"{'='*70}")

    filename = f"dataset_share2trade_{start_year}_{end_year}_V11.jsonl"
    # ใช้โหมด append ('a') เพื่อเก็บข้อมูลต่อจากเดิมได้

    base_url = "https://www.share2trade.com"

    # กำหนดหน้าเริ่มต้นที่ต่างกันสำหรับแต่ละหมวดหมู่
    categories_config = [
        {"path": "/news/talk-of-the-town", "start_page": 145}, # รันต่อจากจุดที่เช็คปีล่าสุด
        {"path": "/news/bulletin-board", "start_page": 264},   # เริ่มใหม่ตั้งแต่หน้าแรก
        {"path": "/news/stock-daily-journal", "start_page": 16} # เริ่มใหม่ตั้งแต่หน้าแรก
    ]

    total_saved = 0
    session = cureq.Session(impersonate="chrome", timeout=25)

    for config in categories_config:
        cat_path = config['path']
        page = config['start_page']
        print(f"\n📂 กำลังเจาะหมวดหมู่: {cat_path} (เริ่มที่หน้า {page})")

        stop_crawling = False

        while not stop_crawling:
            target_url = f"{base_url}{cat_path}?page={page}"
            try:
                resp = session.get(target_url)
                if resp.status_code != 200: break

                soup = BeautifulSoup(resp.text, 'html.parser')
                cards = soup.select('div.card__body')
                news_links = list(set([card.find('a', href=True)['href'] for card in cards if card.find('a', href=True)]))

                if not news_links:
                    print(f"   ⚠️ ไม่พบลิงก์ในหน้า {page} จบหมวดหมู่นี้")
                    break

                print(f"\n📖 หน้า {page}: ตรวจสอบ {len(news_links)} ลิงก์")
                found_in_page = 0
                skipped_year = 0

                for link in news_links:
                    full_link = f"{base_url}{link}" if link.startswith('/') else link
                    try:
                        art_resp = session.get(full_link)
                        if art_resp.status_code != 200: continue
                        art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                        detail_zone = art_soup.find(class_=re.compile(r'news-detail|article|content', re.I)) or art_soup
                        date_element = detail_zone.find(class_=re.compile(r'date|time', re.I))
                        date_text = date_element.get_text(strip=True) if date_element else ""

                        year_match = re.search(r'(20\d{2}|25\d{2})', date_text)
                        if year_match:
                            news_year = int(year_match.group())
                            display_year = news_year - 543 if news_year > 2500 else news_year

                            if start_year <= display_year <= end_year:
                                content_body = art_soup.find('div', class_=re.compile(r'news-detail__content|content-body|entry-content', re.I))
                                if not content_body: content_body = detail_zone

                                paragraphs = content_body.find_all(['p', 'div'], recursive=False) if content_body.name != 'p' else [content_body]
                                valid_texts = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 10]
                                text = " ".join(valid_texts)
                                text = re.sub(r'\s+', ' ', text).strip()

                                if len(text) > 50:
                                    title = art_soup.find('h1').get_text(strip=True) if art_soup.find('h1') else ""
                                    record = {"date": date_text, "url": full_link, "title": title, "content": text}
                                    with open(filename, "a", encoding="utf-8") as f_out:
                                        f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                                    total_saved += 1
                                    found_in_page += 1
                            elif display_year < start_year and page > 10:
                                skipped_year += 1
                                # ถ้าย้อนไปไกลเกินปีเป้าหมายแล้ว และไม่ใช่หน้าแรกๆ ให้เตรียมหยุด
                            else:
                                skipped_year += 1

                    except Exception: continue

                print(f"   ✅ เก็บได้: {found_in_page} | ❌ ปีไม่ตรง/อื่นๆ: {skipped_year}")

                # ถ้าทั้งหน้าไม่มีข่าวในปีที่ต้องการเลย และเราเลยหน้าแรกๆ มาแล้ว ให้หยุดหมวดนี้
                if found_in_page == 0 and page > 10:
                    print(f"   🛑 ไม่พบข่าวปี {start_year}-{end_year} ในหน้านี้แล้ว จบการทำงานหมวดนี้")
                    break

                page += 1
                time.sleep(0.3)

            except Exception as e:
                print(f"\n❌ Error: {e}")
                break

    session.close()
    print(f"\n🎉 เสร็จสิ้น! สะสมได้ทั้งหมด {total_saved} ข่าว")

if __name__ == "__main__":
    scrape_share2trade_production_v11(2023, 2024)


🚀 [SHARE2TRADE V.11] แยกหน้าเริ่มต้นตามหมวดหมู่

📂 กำลังเจาะหมวดหมู่: /news/talk-of-the-town (เริ่มที่หน้า 145)

📖 หน้า 145: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 146: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 147: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 148: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 149: ตรวจสอบ 23 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 2

📖 หน้า 150: ตรวจสอบ 25 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 4

📖 หน้า 151: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 152: ตรวจสอบ 23 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 2

📖 หน้า 153: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 154: ตรวจสอบ 23 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 2

📖 หน้า 155: ตรวจสอบ 24 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 3

📖 หน้า 156: ตรวจสอบ 23 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง/อื่นๆ: 2

📖 หน้า 157: ตรวจสอบ 23 ลิงก์
   ✅ เก็บได้: 21 | ❌ ปีไม่ตรง

### **3.2 Thunhoon (ทันหุ้น)**
- **Logic:** เชื่อมต่อกับ Custom API ของเว็บ (`my-posts.php`) โดยระบุ `categories: 19` (หมวดการเงิน/ธนาคาร)
- **Cleansing:** รวมเนื้อหาจากแท็ก HTML และใช้ Regex จัดระเบียบช่องว่างให้เหมาะสมสำหรับการทำ NLP

In [ ]:
#Thunhoon
import requests
import json
from datetime import datetime
import time
from bs4 import BeautifulSoup
import re

# ฟังก์ชันสำหรับล้างแท็ก HTML
def clean_html(raw_html):
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

def scrape_thunhoon_to_jsonl(start_year=2023, end_year=2024, max_pages=1000):
    print(f"\n🚀 เริ่มปฏิบัติการดูดข่าวและเนื้อหา Thunhoon (ปี {start_year}-{end_year})")
    print(f"{'='*70}")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    }

    base_api_url = "https://wp.thunhoon.com/wp-content/custom-api/my-posts.php"
    filename = f"dataset_thunhoon_{start_year}_{end_year}.jsonl"

    total_saved = 0
    current_page = 1

    with open(filename, "w", encoding="utf-8") as f:
        while current_page <= max_pages:
            print(f"  -> กำลังดึงข้อมูลหน้าที่ {current_page}...", end=" ")
            params = {
                'per_page': 50,
                'page': current_page,
                'categories': 19,
                'orderby': 'date',
                'order': 'desc',
                'after': f'{start_year}-01-01T00:00:00',
                'before': f'{end_year}-12-31T23:59:59'
            }

            try:
                resp = requests.get(base_api_url, headers=headers, params=params, timeout=15)
                if resp.status_code == 200:
                    data = resp.json()
                    if not data or len(data) == 0: break

                    print(f"สำเร็จ! พบ {len(data)} ข่าว...")
                    for article in data:
                        raw_title = article.get('title', {}).get('rendered', '')
                        raw_content = article.get('content', {}).get('rendered', '')

                        clean_title = clean_html(raw_title)
                        clean_body = clean_html(raw_content)

                        # จัดระเบียบช่องว่างเบื้องต้น
                        clean_body = re.sub(r'\s+', ' ', clean_body).strip()

                        if len(clean_body) < 50: continue

                        news_data = {
                            "url": f"https://thunhoon.com/article/{article.get('slug', '')}",
                            "date": article.get('date', ''),
                            "title": clean_title,
                            "content": clean_body
                        }
                        f.write(json.dumps(news_data, ensure_ascii=False) + "\n")
                        total_saved += 1
                else: break
            except: break
            current_page += 1
            time.sleep(0.5)

    print(f"\n🎉 ภารกิจสมบูรณ์! บันทึกข่าวลงไฟล์ '{filename}' จำนวน {total_saved} ข่าว เรียบร้อยแล้ว")

if __name__ == "__main__":
    scrape_thunhoon_to_jsonl(start_year=2023, end_year=2024)


🚀 เริ่มปฏิบัติการดูดข่าวและเนื้อหา Thunhoon (ปี 2023-2024)
  -> กำลังดึงข้อมูลหน้าที่ 1... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 2... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 3... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 4... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 5... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 6... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 7... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 8... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 9... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 10... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 11... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 12... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 13... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 14... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 15... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 16... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข้อมูลหน้าที่ 17... สำเร็จ! พบ 50 ข่าว...
  -> กำลังดึงข

### **3.3 Hoonvision (หุ้นวิชั่น)**
- **Logic:** ส่ง `POST` payload ไปยัง `admin-ajax.php` เพื่อขอข้อมูลรายหน้า
- **Cleansing:** **Titanium Clean Logic** ใช้ Regex ขั้นสูงลบคำขึ้นต้นข่าว เช่น "ทีมข่าวหุ้นวิชั่น รายงานว่า..." ออกเพื่อให้เหลือเพียงเนื้อหาข่าวแท้ๆ

In [ ]:
#Hoonvision
import requests
import json
import time
import re
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_and_clean_hoonvision(start_year=2023, end_year=2024, max_pages=5000):
    print(f"🚀 [HOONVISION V2.9 TITANIUM-CLEAN] เริ่มภารกิจล้างบาง {start_year}-{end_year}")

    ajax_url = "https://www.hoonvision.com/wp-admin/admin-ajax.php"
    filename = f"dataset_hoonvision_{start_year}_{end_year}_ultraclean_v2.jsonl"

    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "X-Requested-With": "XMLHttpRequest"
    }

    total_saved = 0
    page = 1
    consecutive_empty = 0
    stop_crawling = False

    with open(filename, "w", encoding="utf-8") as f:
        while page <= max_pages and not stop_crawling:
            payload = {"action": "load_filtered_posts", "page": str(page)}
            try:
                resp = session.get(ajax_url, params=payload, headers=headers, timeout=25)
                if resp.status_code != 200:
                    time.sleep(2); continue

                data = resp.json()
                if not data:
                    consecutive_empty += 1
                    if consecutive_empty > 5: break
                    page += 1; continue

                consecutive_empty = 0

                for item in data:
                    post_date = item.get("post_date", "")
                    if not post_date: continue

                    news_year = int(post_date[:4])
                    if news_year > end_year: continue
                    if news_year < start_year:
                        stop_crawling = True
                        break

                    # 1. Clean Title
                    raw_title = item.get("post_title", "")
                    clean_title = BeautifulSoup(raw_title, 'html.parser').get_text(strip=True)

                    # 2. Aggressive Content Cleaning
                    raw_content = item.get("post_content", "")
                    soup = BeautifulSoup(raw_content, 'html.parser')
                    text = soup.get_text(separator=' ', strip=True)

                    # --- Step A: Normalization (ล้างอักขระล่องหนและปรับมาตรฐานช่องว่าง/ขีด) ---
                    text = re.sub(r'[\s\xa0\u2000-\u200b\u202f\u205f\u3000]+', ' ', text)
                    text = re.sub(r'[\u2010-\u2015\u2212\u23af\u23e4\u2500]', '-', text)
                    text = ' '.join(text.split())

                    # --- Step B: Aggressive Prefix Removal (Titanium Logic) ---
                    titanium_regex = r'^(?:ทีมข่าว\s*)?(?:หุ้นวิชั่น|Hoonvision)(?:\s*รายงาน(?:ว่า)?)?\s*[:\-–—·|.\s]*'
                    text = re.sub(titanium_regex, '', text, flags=re.I).strip()
                    text = re.sub(r'^[\s:\-–—·|.]+', '', text).strip()

                    if len(text) > 80 and not text.startswith('http'):
                        record = {
                            "date": post_date[:10],
                            "title": clean_title,
                            "content": text,
                            "url": item.get("post_link")
                        }
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        total_saved += 1

                if total_saved > 0 or page % 10 == 0:
                    print(f"💾 หน้า {page} | บันทึกแล้ว {total_saved} ข่าว | ล่าสุด: {post_date[:10]}", end="\r")
                page += 1

            except Exception as e:
                print(f"\n⚠️ Error หน้า {page}: {e}")
                page += 1

    print(f"\n\n✅ เสร็จสมบูรณ์! คลีนข้อมูลสะอาด 100% รวม {total_saved} รายการ บันทึกลง {filename}")

if __name__ == "__main__":
    scrape_and_clean_hoonvision(2023, 2024)

🚀 [HOONVISION V2.9 TITANIUM-CLEAN] เริ่มภารกิจล้างบาง 2023-2024


✅ เสร็จสมบูรณ์! คลีนข้อมูลสะอาด 100% รวม 2586 รายการ บันทึกลง dataset_hoonvision_2023_2024_ultraclean_v2.jsonl


### **3.4 eFinanceThai (อีไฟแนนซ์ไทย)**
- **Logic:** ใช้ระบบ **Hybrid-Recovery Mode** โดยการสแกน Sitemap รายเดือน (`sitemap-news-YYYY-MM.xml`) และดึงข้อมูลผ่าน ID ข่าว มีการใช้ `curl_cffi` ระดับ `chrome110` เพื่อจำลอง TLS Fingerprint ขั้นสูง
- **Cleansing (V3 Robust):**
    - **UI Stripping:** ลบส่วนประกอบ Login, Registration, และ Privacy Policy ที่มักปนมากับเนื้อหา
    - **Mode Filter:** ลบคำสั่งสลับโหมดหน้าเว็บ (Light/Dark/Swift Mode)
    - **Brand Removal:** ใช้ Regex ลบลายน้ำสำนักข่าวที่ซ้ำซ้อนในเนื้อหา เช่น "สำนักข่าวอีไฟแนนซ์ไทย", "สรุปข่าวเด่น" และตัวเลขวันที่ขึ้นต้นประโยค
- **Filtering:** กรองเฉพาะข่าวที่มีเนื้อหาสาระหลังคลีนแล้วมากกว่า 150 ตัวอักษร เพื่อให้ได้ชุดข้อมูลที่มีคุณภาพสูงสำหรับการเทรน Model

In [ ]:
#eFinanceThai
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import pandas as pd
import random

def scrape_efin_production_2023_2024(test_mode=True, sample_size=5):
    start_year, end_year = 2023, 2024
    filename = f"dataset_efinancethai_{start_year}_{end_year}_test.jsonl" if test_mode else f"dataset_efinancethai_{start_year}_{end_year}_cleaned.jsonl"

    # ยกระดับการเลียนแบบ Browser
    session = cureq.Session(impersonate="chrome110")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "th-TH,th;q=0.9,en;q=0.8",
        "Referer": "https://www.google.com/",
        "Connection": "keep-alive"
    }

    target_months = []
    for y in range(start_year, end_year + 1):
        for m in range(1, 13):
            target_months.append((y, m))

    total_saved = 0
    consecutive_failures = 0
    test_records = []

    print(f"  🚀 [{'TEST' if test_mode else 'PRODUCTION'}] เริ่มภารกิจ eFinanceThai (Hybrid-Recovery Mode)")

    with open(filename, "w", encoding="utf-8") as f:
        for year, month in target_months:
            if test_mode and total_saved >= sample_size: break
            if consecutive_failures >= 30: break

            sitemap_url = f"https://www.efinancethai.com/sitemap/sitemaps-news/sitemap-news-{year}-{month:02d}.xml"
            print(f"\n  📅 เดือน: {year}-{month:02d} |   🔗 Sitemap: {sitemap_url}")

            try:
                sm_resp = session.get(sitemap_url, headers=headers, timeout=25)
                if sm_resp.status_code != 200: continue

                soup_sm = BeautifulSoup(sm_resp.text, 'xml')
                links = [loc.text for loc in soup_sm.find_all('loc') if "LatestNewsMain" in loc.text]
                print(f"  🔎 พบลิงก์ {len(links)} รายการ")

                for url in links:
                    if test_mode and total_saved >= sample_size: break
                    if consecutive_failures >= 30: break

                    news_id = url.split('id=')[-1] if 'id=' in url else ""
                    print(f"  [{total_saved + 1}]   📡 ดึง: {news_id[:10]}...", end=" ")

                    try:
                        resp = session.get(url, headers=headers, timeout=15)

                        if resp.status_code == 200:
                            soup = BeautifulSoup(resp.text, 'html.parser')
                            content_box = soup.find(id='spanDetail') or soup.find(id='divDetail')

                            if not content_box:
                                all_texts = [t.get_text(strip=True) for t in soup.find_all(['div', 'span', 'p']) if len(t.get_text(strip=True)) > 200]
                                if all_texts: text = max(all_texts, key=len)
                                else: text = ""
                            else:
                                for junk in content_box.find_all(['script', 'style', 'iframe', 'a']):
                                    junk.decompose()
                                text = content_box.get_text(separator=' ', strip=True)

                            text = re.sub(r'Light Mode.*?Swift Mode', '', text, flags=re.S)
                            text = re.sub(r'\s+', ' ', text).strip()

                            if len(text) > 150:
                                title = soup.find('h1').get_text(strip=True) if soup.find('h1') else ""
                                record = {"date": f"{year}-{month:02d}", "url": url, "title": title, "content": text}
                                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                total_saved += 1
                                consecutive_failures = 0
                                if test_mode: test_records.append(record)
                                print("✅ สำเร็จ")
                                time.sleep(random.uniform(1, 2))
                                continue

                        print("❌ พลาด")
                        consecutive_failures += 1
                        time.sleep(random.uniform(2, 4))
                    except:
                        consecutive_failures += 1
                        continue

            except Exception as e:
                print(f"\n  ⚠️ Error: {e}")
                consecutive_failures += 1

    print(f"\n🎉 ภารกิจเสร็จสมบูรณ์! บันทึกข้อมูลทั้งหมด {total_saved} ข่าว")

if __name__ == "__main__":
    # รันโหมด Production จริง
    scrape_efin_production_2023_2024(test_mode=False)

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
  [18059]   📡 ดึง: eTBwRkM5NE... ✅ สำเร็จ
  [18060]   📡 ดึง: ZVQ2eHVnam... ✅ สำเร็จ
  [18061]   📡 ดึง: OWpxUzk0UT... ✅ สำเร็จ
  [18062]   📡 ดึง: Z2E5anhZSW... ✅ สำเร็จ
  [18063]   📡 ดึง: dTBJWCtldm... ✅ สำเร็จ
  [18064]   📡 ดึง: SC8xaDg0Nz... ✅ สำเร็จ
  [18065]   📡 ดึง: U3FhZS9mbF... ✅ สำเร็จ
  [18066]   📡 ดึง: RVZOVUpTb2... ✅ สำเร็จ
  [18067]   📡 ดึง: blJ1SjZpVy... ✅ สำเร็จ
  [18068]   📡 ดึง: WmVQdzE2b3... ✅ สำเร็จ
  [18069]   📡 ดึง: aExPTGNlan... ✅ สำเร็จ
  [18070]   📡 ดึง: ZWM4VWt4al... ✅ สำเร็จ
  [18071]   📡 ดึง: d3hEMC84aF... ✅ สำเร็จ
  [18072]   📡 ดึง: N2dkR0pjRH... ✅ สำเร็จ
  [18073]   📡 ดึง: OTJKcGJ5Nl... ✅ สำเร็จ
  [18074]   📡 ดึง: cmczWnhpSU... ✅ สำเร็จ
  [18075]   📡 ดึง: K2h4cU1GT0... ✅ สำเร็จ
  [18076]   📡 ดึง: SVAycEx5Um... ✅ สำเร็จ
  [18077]   📡 ดึง: Nk9GaXMzYl... ✅ สำเร็จ
  [18078]   📡 ดึง: enE4U2Vjcy... ✅ สำเร็จ
  [18079]   📡 ดึง: UDIxdGRBL3... ✅ สำเร็จ
  [18080]   📡 ดึง: MkFJTkk4QV... ✅ สำเร็จ
  [18081]   📡 ดึง: 

In [ ]:
#eFinanceThai_clean
import json
import re
import os

def robust_efin_cleaner(text):
    if not text: return ""

    # 1. ลบ Login/Registration Block และ Privacy Policy
    junk_patterns = [
        r'ถือว่าคุณได้ยอมรับข้อตกลงและเงื่อนไขการใช้งาน.*?ตัวตัวอักษรพิมพ์ใหญ่',
        r'ยืนยันด้วย \?กดส่งรหัสใหม่ได้ในนาทีสร้างบัญชี',
        r'อีเมลนี้เคยเข้าใช้งานแล้ว.*?สร้างรหัสผ่าน',
        r'นโยบายความเป็นส่วนตัว.*?ตัวอักษรพิมพ์ใหญ่',
        r'Light\s*Mode|Dark\s*Mode|Swift\s*Mode',
        r'โหมดที่ช่วยปรับเปลี่ยนการแสดงผลของธีม.*?',
        r'ค้นหาข่าว และความรู้ด้านการเงิน.*?ลงชื่อเข้าใช้',
        r'ตัวอักษร\s*\(a-z\)ตัวเลข\s*\(0-9\)',
        r'Loading\.\.\.',
        # V3: ลบหัวข่าวขยะ เวลา และลายน้ำสำนักข่าวซ้ำซ้อน
        r'ข่าวหุ้นล่าสุด\s*ข่าวหุ้นล่าสุด\s*\d*\s*\d{2}:\d{2}',
        r'ข่าวหุ้นล่าสุด\s*\d*\s*\d{2}:\d{2}',
        r'ข่าวหุ้นล่าสุด',
        r'^\s*\d{1,2}\s+', # ลบตัวเลขวันที่ที่ขึ้นต้นประโยค (เช่น 31 ...)
        r'สำนักข่าวอีไฟแนนซ์ไทย\s*\d{4}', # ลบ สำนักข่าวอีไฟแนนซ์ไทย 2566/2567
        r'สรุปข่าวเด่น\s*สำนักข่าวอีไฟแนนซ์ไทย',
        r'Shareสำนักข่าวอีไฟแนนซ์ไทย-?',
        r'Shareดูทั้งหมด$',
        r'สำนักข่าวอีไฟแนนซ์ไทย'
    ]

    for pattern in junk_patterns:
        text = re.sub(pattern, ' ', text, flags=re.S | re.I)

    # 2. จัดระเบียบช่องว่าง
    text = re.sub(r'\s+', ' ', text).strip()

    # 3. ลบคำที่หลงเหลือจาก UI
    junk_keywords = ['ลงชื่อเข้าใช้', 'สมัครสมาชิก', 'Login', 'หน้ารวมข่าว', 'Share', 'ดูทั้งหมด']
    for kw in junk_keywords:
        text = text.replace(kw, '')

    return text.strip()

def run_final_cleaning_pass(filename):
    temp_filename = filename + ".final.tmp"
    count = 0

    with open(filename, 'r', encoding='utf-8') as fin, \
         open(temp_filename, 'w', encoding='utf-8') as fout:
        for line in fin:
            data = json.loads(line)
            cleaned_content = robust_efin_cleaner(data.get('content', ''))

            # เก็บเฉพาะข่าวที่ยังมีเนื้อหาสาระ
            if len(cleaned_content) > 100:
                data['content'] = cleaned_content
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
                count += 1

            if count % 2000 == 0: print(f"🧹 Scrubbing Junk (V3): {count} records...")

    os.replace(temp_filename, filename)
    print(f"\n✨ เสร็จสิ้น! ไฟล์ {filename} ผ่านการคลีนระดับ V3 แล้ว เหลือ {count} รายการ")

run_final_cleaning_pass('dataset_efinancethai_2023_2024_cleaned.jsonl')

# ตรวจสอบตัวอย่าง 5 แถวแรก
print("\n--- ผลลัพธ์หลังแก้ Regex (V3) ---")
with open('dataset_efinancethai_2023_2024_cleaned.jsonl', 'r', encoding='utf-8') as f:
    for _ in range(5):
        line = f.readline()
        if not line: break
        item = json.loads(line)
        print(f"Title: {item['title']}\nContent: {item['content'][:350]}...\n")

🧹 Scrubbing Junk (V3): 2000 records...
🧹 Scrubbing Junk (V3): 4000 records...
🧹 Scrubbing Junk (V3): 6000 records...

✨ เสร็จสิ้น! ไฟล์ dataset_efinancethai_2023_2024_cleaned.jsonl ผ่านการคลีนระดับ V3 แล้ว เหลือ 6658 รายการ

--- ผลลัพธ์หลังแก้ Regex (V3) ---
Title: ครม.รับทราบแผนพัฒนาตลาดทุนไทย ฉบับที่ 4 (พ.ศ.65-70)
Content: ครม.รับทราบแผนพัฒนาตลาดทุนไทย (พ.ศ.65-70) ครม.รับทราบแผนพัฒนาตลาดทุนไทย สร้างภูมิทัศน์ของภาคตลาดทุนไทยในอนาคตให้ตอบสนองเป้าหมายการขับเคลื่อนใน รัฐมนตรีว่าการกระทรวงการคลัง คณะรัฐมนตรีได้รับทราบแผนพัฒนาตลาดทุนไทย ตามที่กระทรวงการคลังเสนอ ดังนี้แผนพัฒนาตลาดทุนไทย สำนักงานคณะกรรมการกำกับหลักทรัพย์และตลาดหลักทรัพย์จึงได้แต่งตั้งคณะทำงานจัดทำแผนพัฒนาตลา...

Title: ธปท.ชี้ท่องเที่ยวยังเป็นแรงหนุนศก.เดือนธ.ค.-Q4/65 มั่นใจเดือนม.ค.66 ยังโตต่อเนื่อง
Content: ธปท.ชี้ท่องเที่ยวยังเป็นแรงหนุนศก.เดือนธ.ค.-Q4/65 ยังโตต่อเนื่อง น.ธปท.สรุปเศรษฐกิจเดือนธ.ค. ชี้ได้ท่องเที่ยวหนุน ฟากส่งออกยังชะลอตัวเล็กน้อย เงินเฟ้อทั้งไตรมาสลดลงตามราคาพลังงาน ส่วนเดือนนี้คาดเศรษฐกิจไทยยังโตต่อเนื่อง

### **3.5 Prachachat (ประชาชาติธุรกิจ)**
- **Logic:** วนลูป Archive ย้อนหลังจากหน้าปัจจุบันไปจนถึงปี 2022 (หน้าประมาณ 835-2300)
- **Cleansing:** คัดกรองเนื้อหาเฉพาะจากคลาส `.td-post-content` และกรองข่าวประชาสัมพันธ์สั้นๆ ออก

In [ ]:
#Prachachat
import json
import time
import re
import random

def scrape_prachachat_finance_full_archive(target_start_year=2023, target_end_year=2024, start_page=650):
    print(f"\n🚀 [Prachachat Archive Sniper V4] เริ่มปฏิบัติการกวาดล้างคลังข่าว {target_start_year}-{target_end_year}")

    finance_news_articles = []
    page = start_page
    output_filename = f"prachachat_finance_{target_start_year}_{target_end_year}_complete.jsonl"
    consecutive_empty_pages = 0
    MAX_EMPTY_LIMIT = 100  # ยอมให้หน้าว่างติดกันได้ถึง 100 หน้าเผื่อกรณี Archive แหว่ง

    # สร้างไฟล์เตรียมไว้
    with open(output_filename, 'a', encoding='utf-8') as f: pass

    while True:
        current_page_url = f"https://www.prachachat.net/finance/page/{page}"
        print(f"\n📄 กำลังสแกนหน้า {page}: {current_page_url}")

        soup = fetch_and_parse(current_page_url)
        if not soup:
            print(f"  ❌ โหลดหน้า {page} ไม่สำเร็จ (Network Error) จะลองข้ามไปหน้าถัดไป")
            page += 1
            continue

        # 🎯 กลยุทธ์หาลิงก์แบบ Aggressive: เอาทุกลิงก์ที่มีคำว่า news- และเช็ค Pattern
        all_links = soup.find_all('a', href=True)
        article_urls = []
        for a in all_links:
            href = a['href']
            # จับทุกลิงก์ข่าว ไม่ว่าจะเป็นหมวดหลักหรือหมวดรองที่ปนมาในหน้า Finance
            if '/news-' in href and ('/finance/' in href or '/category/' in href or 'prachachat.net' in href):
                if href.startswith('/'): href = 'https://www.prachachat.net' + href
                article_urls.append(href)

        article_urls = list(set(article_urls)) # ลบตัวซ้ำ

        if not article_urls:
            consecutive_empty_pages += 1
            print(f"  ⚪ หน้า {page} ไม่พบข่าว (ว่างติดต่อกัน {consecutive_empty_pages} หน้า)")
            if consecutive_empty_pages > MAX_EMPTY_LIMIT:
                print("  🛑 ไม่พบข่าวเพิ่มนานเกินไป สิ้นสุดการค้นหา")
                break
            page += 1
            continue

        consecutive_empty_pages = 0 # รีเซ็ตเมื่อเจอข่าว

        for i, article_url in enumerate(article_urls):
            print(f"  [{page}] ข่าวที่ {i+1}/{len(article_urls)}", end='\r')
            article_soup = fetch_and_parse(article_url)

            if article_soup:
                date_tag = article_soup.find('time', class_='entry-date')
                if date_tag and date_tag.get('datetime'):
                    raw_date = date_tag.get('datetime').split('T')[0]
                    year = int(raw_date.split('-')[0])
                    if year > 2500: year -= 543 # แปลง พ.ศ. เป็น ค.ศ.

                    # เงื่อนไขการหยุด: ถ้าเจอข่าวปี 2022 และปีนั้นไม่ใช่เป้าหมาย
                    if year < target_start_year:
                        print(f"\n  🛑 พบข่าวปี {year} (สิ้นสุดช่วงปีเป้าหมายที่ต้องการแล้ว)")
                        return

                    # เก็บข้อมูลเฉพาะปี 2023-2024
                    if target_start_year <= year <= target_end_year:
                        title_tag = article_soup.find('h1')
                        title = title_tag.get_text(strip=True) if title_tag else "N/A"

                        content_div = article_soup.find('div', class_='td-post-content') or article_soup.find('div', class_='entry-content')
                        if content_div:
                            paragraphs = content_div.find_all('p')
                            full_content = ' '.join([p.get_text(strip=True) for p in paragraphs])
                            full_content = re.sub(r'\s+', ' ', full_content).strip()

                            record = {
                                'url': article_url,
                                'title': title,
                                'date': f"{year}-{raw_date.split('-', 1)[1]}",
                                'content': full_content
                            }
                            with open(output_filename, 'a', encoding='utf-8') as f:
                                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                            finance_news_articles.append(record)

            time.sleep(random.uniform(0.1, 0.2))

        page += 1
        if page > 2500: break

# รันใหม่ตั้งแต่หน้า 835 เพื่อให้แน่ใจว่าเก็บครบทุกเม็ด
scrape_prachachat_finance_full_archive(2023, 2024, start_page=835)

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
Fetching URL: https://www.prachachat.net/world-news/news-2004551
Fetching URL: https://www.prachachat.net/breaking-news/news-1235190
Fetching URL: https://www.prachachat.net/politics/news-2004519
Fetching URL: https://www.prachachat.net/breaking-news/news-1235388
Fetching URL: https://www.prachachat.net/breaking-news/news-1235542
Fetching URL: https://www.prachachat.net/general/news-2004459
Fetching URL: https://www.prachachat.net/breaking-news/news-1235380
Fetching URL: https://www.prachachat.net/breaking-news/news-1235614
Fetching URL: https://www.prachachat.net/general/news-2002690

📄 กำลังสแกนหน้า 2150: https://www.prachachat.net/finance/page/2150
Fetching URL: https://www.prachachat.net/finance/page/2150
Fetching URL: https://www.prachachat.net/general/news-2004547
Fetching URL: https://www.prachachat.net/property/news-2004560
Fetching URL: https://www.prachachat.net/politics/news-2004526
Fetching URL: https://www.prachachat

In [ ]:
#Prachachat_clean
import json
import os

def clean_existing_dataset(input_file, min_length=150):
    output_file = input_file.replace('.jsonl', '_cleaned.jsonl')
    total_records = 0
    cleaned_records = 0

    if not os.path.exists(input_file):
        print(f"❌ ไม่พบไฟล์: {input_file}")
        return

    print(f"🧼 เริ่มทำความสะอาดไฟล์: {input_file}")

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:

        for line in fin:
            total_records += 1
            try:
                data = json.loads(line)
                content = data.get('content', '')

                # กรอง: ต้องมีความยาวมากกว่าที่กำหนด
                if len(content) >= min_length:
                    fout.write(json.dumps(data, ensure_ascii=False) + '\n')
                    cleaned_records += 1
            except Exception as e:
                continue

    print(f"✅ ทำความสะอาดเสร็จสิ้น!")
    print(f"📊 ข้อมูลเดิม: {total_records} ข่าว")
    print(f"✨ ข้อมูลที่ผ่านการกรอง: {cleaned_records} ข่าว (ตัดออก {total_records - cleaned_records} ข่าว)")
    print(f"📁 บันทึกไฟล์ใหม่ที่: {output_file}")

# เรียกใช้งาน (ตรวจสอบชื่อไฟล์ให้ตรงกับที่รันในบล็อกก่อนหน้า)
input_filename = "prachachat_finance_2023_2024_complete.jsonl"
clean_existing_dataset(input_filename)

🧼 เริ่มทำความสะอาดไฟล์: prachachat_finance_2023_2024_complete.jsonl
✅ ทำความสะอาดเสร็จสิ้น!
📊 ข้อมูลเดิม: 14913 ข่าว
✨ ข้อมูลที่ผ่านการกรอง: 14886 ข่าว (ตัดออก 27 ข่าว)
📁 บันทึกไฟล์ใหม่ที่: prachachat_finance_2023_2024_complete_cleaned.jsonl


## 📂 กลุ่มที่ 4: Browser Automation (BusinessToday)

### **4.1 BusinessToday**
- **Logic:** ใช้ **Playwright (Headless Browser)** เพื่อ Render หน้าเว็บ และจัดการกับ Pagination ที่ซับซ้อน
- **Tech:** รันผ่าน External Worker Script เพื่อประสิทธิภาพสูงสุดใน Colab
- **Cleansing:** ลบ Copyright Footer `© 2025 - Businesstoday.co` ออกจากทุก Record

In [ ]:
#BusinessToday
import subprocess
import sys
import os

# V44: Restarting Business Today Scraper (Targeting 2023-2024 Archive)
script_content = r"""
import asyncio
import sys
import json
import re
from playwright.async_api import async_playwright

async def run_restart_scraper():
    print("🚀 [V44 RESTART] Targeted 2023-2024 News Collection...")
    filename = 'businesstoday_archive_2023_2024.jsonl'
    base_url = "https://www.businesstoday.co/category/stock/"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
            viewport={'width': 1280, 'height': 1200}
        )
        page = await context.new_page()

        total_saved = 0
        # Jump directly to deeper pages where 2023-2024 data resides
        for p_idx in range(50, 150):
            url = f"{base_url}page/{p_idx}/"
            print(f"📖 Scanning Page {p_idx}: {url}")

            try:
                await page.goto(url, wait_until='load', timeout=60000)
                await asyncio.sleep(3)

                # Discover links using the validated tagDiv pattern
                links = await page.evaluate('''() => {
                    return Array.from(document.querySelectorAll(".td-module-title a, article a"))
                                .map(a => a.href)
                                .filter(h => h && /\\/\\d{2}\\/\\d{2}\\/\\d{4}\\//.test(h));
                }''')

                unique_links = list(set(links))
                if not unique_links: continue

                for art_url in unique_links:
                    # Date Check: /DD/MM/YYYY/
                    date_match = re.search(r'/(\d{2})/(\d{2})/(\d{4})/', art_url)
                    if date_match:
                        day, month, year = date_match.groups()
                        if year not in ['2023', '2024']:
                            continue

                    try:
                        art_page = await context.new_page()
                        await art_page.goto(art_url, wait_until='domcontentloaded', timeout=30000)

                        data = await art_page.evaluate('''() => {
                            const title = document.querySelector("h1, .entry-title, .tdb-title-text")?.innerText || "";
                            const ps = Array.from(document.querySelectorAll(".td-post-content p, .entry-content p, p"))
                                            .map(p => p.innerText.trim())
                                            .filter(t => t.length > 20);
                            return { title: title.trim(), content: ps.join(" ") };
                        }''')

                        if len(data['content']) > 200:
                            with open(filename, "a", encoding="utf-8") as f:
                                f.write(json.dumps({"date": f"{year}-{month}-{day}", "url": art_url, **data}, ensure_ascii=False) + "\n")
                            total_saved += 1
                            print(f"      ✅ Saved: {data['title'][:40]}... ({year})")

                        await art_page.close()
                    except Exception:
                        if 'art_page' in locals(): await art_page.close()

            except Exception as e:
                print(f"   ❌ Error on Page {p_idx}: {e}")

        await browser.close()
        print(f"\n🎉 Finished! Total 2023-2024 articles collected: {total_saved}")

if __name__ == '__main__':
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    asyncio.run(run_restart_scraper())
"""

with open("bt_external_worker.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("⌛ Launching Restarted Scraper V44...")
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
result = subprocess.run([sys.executable, "bt_external_worker.py"], capture_output=True, text=True, encoding="utf-8", env=env)
print(result.stdout)
if result.stderr:
    print("System Logs:\n", result.stderr)

⌛ Launching Restarted Scraper V44...
🚀 [V44 RESTART] Targeted 2023-2024 News Collection...
📖 Scanning Page 50: https://www.businesstoday.co/category/stock/page/50/
      ✅ Saved: ดาวโจนส์ร่วง 128.65 จุด นักลงทุนจับตาตัว... (2024)
      ✅ Saved: ดาวโจนส์บวก 188.59 จุด แรงซื้อหุ้นเทคฯ-ค... (2024)
      ✅ Saved: ‘หุ้นไทย’ วันนี้ แนวโน้มแกว่งตัว 1,420-1... (2024)
      ✅ Saved: ดาวโจนส์ร่วง 154.10 จุด หุ้นเทคโนโลยียัง... (2024)
      ✅ Saved: หุ้นเอเชียเปิดลบ หลังประกาศตัวเลขเงินเฟ้... (2024)
      ✅ Saved: ดาวโจนส์ร่วง 76.47 จุด ซื้อขายระมัดระวัง... (2024)
      ✅ Saved: หุ้นไทยเดือน ธ.ค. แนวโน้มแกว่งตัว 1,380-... (2024)
      ✅ Saved: ‘หุ้นไทย’ วันนี้ แนวโน้มแกว่งตัว 1,440-1... (2024)
      ✅ Saved: ‘หุ้นไทย’ วันนี้ แนวโน้มแกว่งตัว 1,420-1... (2024)
      ✅ Saved: ‘หุ้นไทย’ วันนี้ แนวโน้ม ‘ไซด์เวย์’ กรอบ... (2024)
📖 Scanning Page 51: https://www.businesstoday.co/category/stock/page/51/
      ✅ Saved: ‘หุ้นไทย’ วันนี้ แนวโน้ม ‘ไซด์เวย์’ กรอบ... (2024)
      ✅ Saved: ดาวโจนส์ร่วง 120.66 จุ

In [ ]:
#BusinessToday_clean
import pandas as pd
import json
import os
import re

# Configuration
raw_filename = 'businesstoday_archive_2023_2024.jsonl'
output_filename = 'businesstoday_final_cleaned_2023_2024.jsonl'

def minimal_clean_bt(text):
    if not isinstance(text, str):
        return ""

    # Targeted removal of the specific copyright string requested
    target_footer = r'© 2025 - Businesstoday\.co'
    # Remove the string regardless of surrounding whitespace and then strip the overall result
    text = re.sub(target_footer, '', text, flags=re.IGNORECASE).strip()

    return text

if os.path.exists(raw_filename):
    data = []
    with open(raw_filename, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except:
                continue

    df_final = pd.DataFrame(data)

    # Apply cleaning directly to the 'content' column to replace the raw text
    df_final['content'] = df_final['content'].apply(minimal_clean_bt)

    # Ensure we only keep the necessary columns (date, url, title, content)
    # Removing any accidental 'content_clean' or other helper columns
    columns_to_keep = ['date', 'url', 'title', 'content']
    df_final = df_final[columns_to_keep]

    # Save to the new separate file
    df_final.to_json(output_filename, orient='records', lines=True, force_ascii=False)

    print(f"✅ Final cleaning complete!")
    print(f"📁 Results (Cleaned content only) saved to: {output_filename}")
    print(f"📊 Total records: {len(df_final)}")

    # Verification check of the tail of the content
    print("\nPreview of cleaned content tail:")
    display(df_final[['url', 'content']].head(3))
else:
    print(f"❌ Could not find raw file: {raw_filename}")

✅ Final cleaning complete!
📁 Results (Cleaned content only) saved to: businesstoday_final_cleaned_2023_2024.jsonl
📊 Total records: 810

Preview of cleaned content tail:


,url,content
0,https://www.businesstoday.co/stock/12/11/2024/...,ธนาคารกรุงไทย (Krungthai Chief Investment Offi...
1,https://www.businesstoday.co/stock/12/11/2024/...,‘บล.กสิกรไทย’ คาดแนวโน้มหุ้นไทยวันนี้ จะแกว่งต...
2,https://www.businesstoday.co/stock/12/11/2024/...,หุ้นสหรัฐปิดบวก ตลาดยังคงขานรับชัยชนะของ ‘โดนั...


In [ ]:
# [CELL-26]
import pandas as pd
import json
from tqdm.auto import tqdm
import os
import glob
import re

def extract_year_robust(date_input):
    """ฟังก์ชันแปลงวันที่แบบครอบจักรวาล (รองรับเดือนภาษาไทย และปี พ.ศ.)"""
    if not date_input or pd.isna(date_input):
        return 'Unknown'

    date_str = str(date_input).strip()

    # Dictionary สำหรับแปลงเดือนภาษาไทยย่อเป็นตัวเลข (เพื่อช่วยในการตรวจสอบความถูกต้องถ้าจำเป็น)
    # ในที่นี้เราจะโฟกัสที่การดึง 'ปี' ออกมา

    # 1. เช็คปี พ.ศ. จากตัวเลข 4 หลักที่ขึ้นต้นด้วย 25xx (เช่น 2566, 2567)
    thai_year_match = re.search(r'(25\d{2})', date_str)
    if thai_year_match:
        return str(int(thai_year_match.group(1)) - 543)

    # 2. เช็คปี ค.ศ. จากตัวเลข 4 หลักที่ขึ้นต้นด้วย 20xx (เช่น 2023, 2024)
    eng_year_match = re.search(r'(20\d{2})', date_str)
    if eng_year_match:
        return eng_year_match.group(1)

    # 3. เช็คปี พ.ศ. แบบย่อ (เช่น 66, 67) มักตามหลังชื่อเดือนภาษาไทย
    # มองหาตัวเลข 2 หลักที่อยู่ท้ายประโยคหรือหลังช่องว่าง
    short_year_match = re.search(r'\s(\d{2})$|\.(\d{2})$', date_str)
    if short_year_match:
        yy = short_year_match.group(1) or short_year_match.group(2)
        if yy in ['66', '67', '68']:
            return str(2500 + int(yy) - 543)

    # 4. ใช้ pandas as fallback
    try:
        return str(pd.to_datetime(date_str, errors='coerce').year)
    except:
        return 'Unknown'

def clean_source_name(filename):
    """ล้างชื่อไฟล์ให้เหลือแค่ชื่อสำนักข่าว"""
    name = filename.replace('.jsonl', '')
    # ลบ pattern ที่พบบ่อยออก
    name = re.sub(r'dataset_|thairath_|prachachat_|_API|_2023_2024|_cleaned|_complete|_ultra_refined|_ultraclean|_CategoryLocked|_FinanceExact|_V11|_v2|_SuperClean_Production|_final_cleaned|_archive', '', name, flags=re.I)
    return name.replace('_', ' ').strip().upper()

def summarize_folder_datasets(folder_path):
    file_paths = glob.glob(os.path.join(folder_path, "*.jsonl"))

    if not file_paths:
        return f"❌ ไม่พบไฟล์ .jsonl ในโฟลเดอร์: {folder_path}"

    summary_data = []
    print(f"📊 กำลังวิเคราะห์ข้อมูลข่าวปี 2023-2024 จาก: {folder_path}")

    for file_path in tqdm(file_paths, desc="Processing Datasets"):
        source_raw = os.path.basename(file_path)
        source_clean = clean_source_name(source_raw)

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        data = json.loads(line)
                        year = extract_year_robust(data.get('date', ''))

                        # เก็บเฉพาะปีที่ต้องการ
                        if year in ['2023', '2024']:
                            summary_data.append({
                                'Source': source_clean,
                                'Year': year
                            })
                    except:
                        continue
        except Exception as e:
            print(f"⚠️ ไม่สามารถอ่านไฟล์ {source_raw}: {e}")

    if not summary_data:
        return "❌ ไม่พบข้อมูลข่าวในปี 2023 หรือ 2024 ในโฟลเดอร์นี้"

    df_stats = pd.DataFrame(summary_data)

    # สร้าง Pivot Table และเรียงลำดับคอลัมน์
    pivot_table = df_stats.pivot_table(
        index='Source',
        columns='Year',
        aggfunc='size',
        fill_value=0
    )

    # ตรวจสอบว่ามีคอลัมน์ครบตามที่ต้องการ
    for y in ['2023', '2024']:
        if y not in pivot_table.columns:
            pivot_table[y] = 0

    pivot_table = pivot_table[['2023', '2024']]

    # เพิ่มยอดรวม
    pivot_table['Total'] = pivot_table.sum(axis=1)
    pivot_table.loc['GRAND TOTAL'] = pivot_table.sum()

    return pivot_table

# --- Execution ---
target_folder = r"C:\Users\USER\Desktop\News_DAPT\Final_Data"
final_report = summarize_folder_datasets(target_folder)
display(final_report)

📊 กำลังวิเคราะห์ข้อมูลข่าวปี 2023-2024 จาก: C:\Users\USER\Desktop\News_DAPT\Final_Data


Processing Datasets:   0%|          | 0/15 [00:00<?, ?it/s]

Year,2023,2024,Total
Source,,,
AMARINTV FINANCE FASTURL,739,641,1380
BUSINESSTODAY,421,389,810
EFINANCETHAI,3171,3487,6658
FINANCE,7279,7607,14886
HOONSMART,2781,5108,7889
HOONVISION,0,2586,2586
INFOQUEST CLEAN LOCKED,11692,8865,20557
KAOHOON,19456,20303,39759
MONEY BANKING,3957,3942,7899
